# CUBIC MARS PS3 -- Source-First Root-Cause and Severity Intelligence V26

## SageMaker edition -- direct PS1 Silver OOS facts, optional PS3 labels, device/component/causal dashboard facts

V24 follows a V23 run that completed cleanly but published five one-row placeholder
tables. The source export turned out to be **76 columns wide**, not the narrow export
assumed since V22, and two of my own defects were keeping that width out of the pipeline.
The OOS contract is unchanged across all four versions. The OOS
contract is unchanged and deliberately so: the spine is still PS1-compatible
hardware-OOS SET events read from the direct Silver export, never from Gold and
never from chargeability.

### What this notebook does

```text
PS1 Silver device_event_enriched
  hardware OOS + SET + TVM/GATE/VALIDATOR + current devices
        v
PS1-compatible device OOS episode spine (3-day gap sessions)
        v
optional PS2 feature snapshots and PS3 label/evidence exports
        v
observed severity / confirmed root cause / leakage-safe per-category models
        v
bounded observational causal layer (cross-fitted AIPW)
        v
19 device, component, facility, serial and causal Parquet tables in S3
        v
run_complete.json -> Lambda loader -> Aurora -> API -> dashboard
```

### What changed from V21, and why

| Area | V21 | V22 |
|---|---|---|
| Engine | Spark only. The attached V21 run returned `not_ready_oos_source` because Java was missing, and produced no facts at all | pyarrow spine first, Spark as fallback. Identical OOS semantics on both routes, verified against a row-by-row reference implementation |
| Pre-event features | Per-device Python loop writing into the full frame each iteration | Single sorted pass, one assignment. 63x faster on 20k episodes, output identical |
| Models | Three pooled models | Per device category, with a clearly labelled pooled fallback only for a category that cannot meet its own support floor |
| Causal | Dropped entirely | Bounded cross-fitted AIPW with balance diagnostics, overlap and Holm-Bonferroni multiplicity control |
| Run id | `ps3_20260802T135023Z_...`, which the loader's `run_id=([0-9a-f-]+)` pattern rejects | Sortable hex-epoch UUID that both sorts chronologically and matches the loader |
| Overflow | Above the episode cap, published no dashboard table at all | Keeps the most recent window that fits and stamps `episode_scope_status` on every row |
| Scan (V23) | Read the whole scan in one call behind a row cap. A real run died at 40,238,324 candidate rows against a cap of 40,000,000, then had no Spark to fall back to | Streams record batches with the PS1 predicate pushed into the scan. Peak memory is one batch plus surviving rows, so source size is no longer a failure mode |
| Arrow projection (V23) | The pyarrow route silently dropped `facility_id`, `bus_id` and `component_serial_nbr`, so the facility and serial tables would read "unavailable" on every real run | All three carried on both routes |
| Failure reporting (V23) | Cell 9 printed only fact tables, so a failed run showed no reason | The audit tables print on every path |
| Component alias (V24) | The alias list normalised to `componenttype` and the export calls it `COMPONENT_TYPE_NAME`, so every episode had a null component -- which emptied attribution, the repeat-interval table and the entire causal layer | Aliases matched against the real schema; one `PS3V22_COLUMN_ALIASES` table so a gap is visible in one place |
| Episode aggregation (V24) | The pandas sessioniser carried six fields, so `facility_id` and `component_serial_nbr` existed on the event frame and were dropped at episode grain -- `ps3_facility_rollup` and `ps3_serial_reliability` published one placeholder row on a successful run | Facility, bus, serial, position, subsystem, event type and severity all carried |
| Downtime (V24) | Not computed | `oos_minutes_union` per episode -- the union of the [event, clear] intervals, never their sum, with the naive sum published beside it. This is the exact defect being investigated in PS2, avoided here by construction |
| Causal estimator (V25) | The propensity model was fitted with `class_weight="balanced"`. Balancing distorts predicted probabilities toward the minority class, which is right for a recall-tuned classifier and wrong for a propensity score, because AIPW divides by it. On the real run it pushed propensities on treated units to ~0.60 where true prevalence was ~1.2%, collapsing the inverse-probability weights and **understating standard errors by roughly 3x**. On a simulated no-effect case at 2.3% prevalence it produced a false positive at \|z\|=2.54 | Unweighted propensity model, standardised features, `max_iter=2000`, and overlap measured against the common-support region actually shared by the two arms rather than a fixed 0.02-0.98 band that declares near-zero overlap for any rare treatment |
| Loadable keys (V26) | Two tables carried a natural key that is entirely null when the PS3 evidence lane is unconfigured -- `ps3_component_summary.dashboard_root_cause_domain`, and `oos_episode_id`/`feature` on `ps3_prediction_explainability`. The RDS loader refuses a table whose declared key is null, so both would have failed on first load | Key columns are sentinel-filled at the publish boundary, which covers every producer rather than each one separately, and the explainability key is de-duplicated |
| Source labels (V24) | Severity came only from the optional PS3 Gold export, so every model was skipped for "0 labelled rows" | `ps3_source_column_profile` measures each candidate label column and promotes source severity only when it is **not** fully determined by `EVENT_TYPE_ID`; a column that is a lookup on the event type is refused |
| Output | Gold bucket, no loader watching it | Artifacts bucket alongside PS2 and PS4, with `run_complete.json` written only for a complete run |
| Tables | 9 plus control | 19, adding device-day, facility, device and serial reliability, repeat interval, commanded split, causal effects, causal balance and run status |
| Commanded OOS | In the spine, invisible downstream | In the spine (so PS3 reconciles with PS1 and PS2) and split out in every summary so the dashboard can exclude planned work |

Function names keep the `ps3v21_` prefix on purpose, so this notebook diffs
cleanly against V21 line by line. Only new functions use `ps3v22_`.

### Non-negotiable semantic contract -- unchanged from V21

| Subject | This notebook does | This notebook does not do |
|---|---|---|
| OOS discovery | Reads PS1-compatible hardware-OOS `SET` events from the direct Silver export or a bounded verified export | Never discovers OOS events from PS3 Gold or chargeability fields |
| Severity | Uses linked observed/approved severity only; otherwise records it as unavailable | Never invents a physical severity label |
| Root cause | Confirms cause only with linked evidence **and** approved taxonomy mapping | Never calls SHAP, raw event text, or a component field a confirmed root cause |
| Causal layer | Reports observational, cross-fitted, multiplicity-controlled effects with their diagnostics | Never claims an experiment, and never presents an effect as proof of causation |
| Validators | Publishes Validator OOS facts even when labels are absent | Never manufactures Validator cause or severity labels |
| Historical dump | Marks the 11-Apr-2026 snapshot as historical and right-censored | Never presents it as a current operational score |


## Before running

Run the cells in order. The only normal configuration work is in the next cell.

**Prove it first.** Set `PS3_SMOKE_TEST_SYNTHETIC=true` and Run All. That builds a
realistic synthetic spine and exercises every lane -- features, per-category
models, SHAP, the causal layer and all 19 tables -- without Spark, without S3 and
without AWS credentials. Synthetic mode forces `PUBLISH_TO_S3` off, so nothing
can reach a real prefix. V21's synthetic path had four rows, which is below every
model support floor, so it could only prove the notebook started.

**No Java needed.** The spine is read with pyarrow, streamed in batches. The PS3 space
has no Java and does not need one; Spark remains only as an unused fallback.

Defaults are usable in the Chicago SageMaker environment as they stand:

- Direct Silver OOS discovery: `.../chicago/silver/device_event_enriched/`
- Current-device filter matching PS1: `.../chicago/silver/dim_device/`
- Dashboard output: `s3://...-artifacts-.../ps3_outputs/`

`PS3_RUN_MODE` defaults to `REPLAY`, which is forced onto an isolated prefix and
emits no loader trigger. Set it to `PRODUCTION` only for the run you intend the
dashboard to pick up.

Optional exports are used only when explicitly configured. `PS3_GOLD_INCIDENT_LABEL_EXPORT`
is a **label/evidence** lane; it is never used to find or filter the OOS spine.

For the historical dump, retain `PS3_DATA_AS_OF_DATE=2026-04-11`. When a current
governed snapshot arrives, update only that date and the source paths, then
rebuild the models. Do not make current operational decisions from this run.


In [1]:
import os

# Job-supplied env WINS; these are defaults, so an unparameterised run behaves
# exactly as before. A scheduled Processing job passes PS3_DATA_AS_OF_DATE for the
# run's vintage -- a hard assignment here would silently overwrite it and every
# scheduled run would re-publish the same April window under a fresh run_id.
os.environ.setdefault("PS3_RUN_MODE",         "PRODUCTION")
os.environ.setdefault("PS3_S3_OUTPUT_PREFIX", "s3://cubic-mars-pm-s3-datalake-dev-artifacts-170202974600/chicago/ps3_outputs")
os.environ.setdefault("PS3_DATA_AS_OF_DATE",  "2026-04-11")

for k in ("PS3_RUN_MODE", "PS3_S3_OUTPUT_PREFIX", "PS3_DATA_AS_OF_DATE"):
    print(f"{k:22} = {os.environ[k]}")


PS3_RUN_MODE           = PRODUCTION
PS3_S3_OUTPUT_PREFIX   = s3://cubic-mars-pm-s3-datalake-dev-artifacts-170202974600/chicago/ps3_outputs
PS3_DATA_AS_OF_DATE    = 2026-04-11


In [2]:
# 1. Configuration — edit only this cell when wiring new governed exports.
# V22. The OOS contract is unchanged from V21: hardware OOS + SET, from the
# PS1 direct-Silver export, never from Gold and never from chargeability.
import os
import json
import re
import uuid
import traceback
import datetime as dt
from pathlib import Path


def _ps3v21_env(name, default=""):
    value = os.environ.get(name, default)
    return str(value).strip() if value is not None else ""


def _ps3v21_bool(value, default=False):
    if value is None or str(value).strip() == "":
        return bool(default)
    return str(value).strip().lower() in {"1", "true", "t", "yes", "y", "on"}


PS3V21_CONFIG = {
    # Historical snapshot policy. This must be the last fully governed date in
    # the client dump, not today's notebook execution date.
    "DATA_AS_OF_DATE": _ps3v21_env("PS3_DATA_AS_OF_DATE", "2026-04-11"),
    "LOOKBACK_DAYS": int(_ps3v21_env("PS3_LOOKBACK_DAYS", "730")),
    "PS1_HORIZON_DAYS": int(_ps3v21_env("PS3_PS1_HORIZON_DAYS", "3")),
    "PS1_OOS_EPISODE_GAP_DAYS": int(_ps3v21_env("PS3_PS1_OOS_EPISODE_GAP_DAYS", "3")),
    "PS1_CURRENT_DEVICE_FILTER": _ps3v21_bool(_ps3v21_env("PS3_PS1_CURRENT_DEVICE_FILTER", "true"), True),
    # Kept False by default to reproduce the original PS1 hardware-OOS SET
    # definition faithfully. Chargeability is never read or used.
    "EXCLUDE_COMMANDED_OOS": _ps3v21_bool(_ps3v21_env("PS3_EXCLUDE_COMMANDED_OOS", "false"), False),

    # Source priority: bounded direct event export -> verified episode export
    # -> direct Silver bootstrap. `auto` is safe: unavailable sources produce
    # a report-only result, not an exception.
    "SOURCE_MODE": _ps3v21_env("PS3_SOURCE_MODE", "auto").lower(),
    "PS1_DIRECT_EVENT_EXPORT": _ps3v21_env("PS3_PS1_DEVICE_EVENT_EXPORT") or _ps3v21_env("PS3_PS1_DEVICE_EVENT_ENRICHED_EXPORT"),
    "PS1_VERIFIED_EPISODE_EXPORT": _ps3v21_env("PS3_PS1_OOS_EPISODE_EXPORT"),
    "ALLOW_DIRECT_SILVER_BOOTSTRAP": _ps3v21_bool(_ps3v21_env("PS3_ALLOW_DIRECT_SILVER_BOOTSTRAP", "true"), True),
    "RAW_SILVER_DEVICE_EVENT_PATH": _ps3v21_env(
        "PS3_SILVER_DEVICE_EVENT_PATH",
        "s3://cubic-mars-pm-s3-datalake-dev-gold-170202974600/chicago/silver/device_event_enriched/",
    ),
    "RAW_SILVER_DIM_DEVICE_PATH": _ps3v21_env(
        "PS3_SILVER_DIM_DEVICE_PATH",
        "s3://cubic-mars-pm-s3-datalake-dev-gold-170202974600/chicago/silver/dim_device/",
    ),

    # Optional enrichment only. They never determine whether an OOS episode
    # exists. Leave blank until a governed, bounded export is available.
    "PS1_FEATURE_EXPORT": _ps3v21_env("PS3_PS1_FEATURE_EXPORT"),
    "PS2_FEATURE_EXPORT": _ps3v21_env("PS3_PS2_FEATURE_EXPORT"),
    "GOLD_INCIDENT_LABEL_EXPORT": _ps3v21_env("PS3_GOLD_INCIDENT_LABEL_EXPORT") or _ps3v21_env("PS3_GOLD_INCIDENT_EXPORT"),
    "INCIDENT_ROOT_CAUSE_EXPORT": _ps3v21_env("PS3_INCIDENT_ROOT_CAUSE_EXPORT"),
    "MAINTENANCE_LEDGER_EXPORT": _ps3v21_env("PS3_MAINTENANCE_LEDGER_EXPORT"),
    "INCIDENT_TASK_CI_EXPORT": _ps3v21_env("PS3_INCIDENT_TASK_CI_EXPORT"),
    "ROOTCAUSE_TAXONOMY_EXPORT": _ps3v21_env("PS3_ROOTCAUSE_TAXONOMY_EXPORT"),
    "EVIDENCE_LINK_TOLERANCE_HOURS": int(_ps3v21_env("PS3_EVIDENCE_LINK_TOLERANCE_HOURS", "72")),

    # Scale controls. These never silently sample a dashboard fact table.
    # If the episode table exceeds this local-model limit, full Spark facts are
    # published and model/evidence enrichment is reported as deferred.
    "MAX_LOCAL_EPISODES": int(_ps3v21_env("PS3_MAX_LOCAL_EPISODES", "2000000")),
    "MAX_OPTIONAL_EXPORT_ROWS": int(_ps3v21_env("PS3_MAX_OPTIONAL_EXPORT_ROWS", "300000")),
    "MAX_SHAP_EPISODES": int(_ps3v21_env("PS3_MAX_SHAP_EPISODES", "1000")),
    "MIN_TRAIN_ROWS": int(_ps3v21_env("PS3_MIN_TRAIN_ROWS", "120")),
    "MIN_TRAIN_CLASS_ROWS": int(_ps3v21_env("PS3_MIN_TRAIN_CLASS_ROWS", "12")),
    "MIN_VALID_CLASS_ROWS": int(_ps3v21_env("PS3_MIN_VALID_CLASS_ROWS", "4")),
    "MIN_MACRO_F1": float(_ps3v21_env("PS3_MIN_MACRO_F1", "0.45")),
    "MIN_MACRO_F1_LIFT": float(_ps3v21_env("PS3_MIN_MACRO_F1_LIFT", "0.05")),
    "MIN_BALANCED_ACCURACY": float(_ps3v21_env("PS3_MIN_BALANCED_ACCURACY", "0.40")),

    # Immutable standard publication. A run overwrites only its own exact
    # run-id partition; it never deletes a broad production prefix and emits
    # no RDS trigger manifest.
    "PUBLISH_TO_S3": _ps3v21_bool(_ps3v21_env("PS3_PUBLISH_TO_S3", "true"), True),
    "S3_OUTPUT_PREFIX": _ps3v21_env(
        "PS3_S3_OUTPUT_PREFIX",
        # V22: artifacts bucket, matching PS2 and PS4, so the existing Lambda
        # loader and its IAM scope work unchanged. No Gold object is written.
        "s3://cubic-mars-pm-s3-datalake-dev-artifacts-170202974600/ps3_outputs",
    ).rstrip("/"),
    "S3_BOOTSTRAP_PREFIX": _ps3v21_env(
        "PS3_S3_BOOTSTRAP_PREFIX",
        "s3://cubic-mars-pm-s3-datalake-dev-artifacts-170202974600/chicago/ps3_inputs/ps1_hardware_oos_set_events",
    ).rstrip("/"),
    "WRITE_BOUNDED_SOURCE_EXPORT": _ps3v21_bool(_ps3v21_env("PS3_WRITE_BOUNDED_SOURCE_EXPORT", "true"), True),
    "LOCAL_OUTPUT_ROOT": _ps3v21_env("PS3_LOCAL_OUTPUT_ROOT", "PS3_V26_outputs"),

    # Never enable this for a real delivery. It exists only to prove notebook
    # mechanics in an isolated development environment.
    "SMOKE_TEST_SYNTHETIC": _ps3v21_bool(_ps3v21_env("PS3_SMOKE_TEST_SYNTHETIC", "false"), False),
    # --- V22 additions -----------------------------------------------------
    # Run mode. A REPLAY run is forced onto an isolated prefix so a historical
    # rebuild can never land on the prefix the RDS loader watches.
    "RUN_MODE": _ps3v21_env("PS3_RUN_MODE", "REPLAY").upper(),

    # Engine. The spine is read with pyarrow first, which removes the JVM
    # dependency that made the V21 run return not_ready_oos_source. Spark stays
    # as the fallback and the OOS semantics are identical on both routes.
    "PREFER_PYARROW_SPINE": _ps3v21_bool(_ps3v21_env("PS3_PREFER_PYARROW_SPINE", "true"), True),
    # V23: the reader streams record batches, so peak memory is one batch plus
    # the rows that survive the PS1 predicate. Size is no longer a failure mode.
    # V22 read the scan in one call and guarded it with a row cap; a real run
    # died at 40,238,324 candidate rows against a cap of 40,000,000 -- 0.6% over
    # an arbitrary limit. These two remain only as runaway guards.
    "PYARROW_MAX_SCAN_ROWS": int(_ps3v21_env("PS3_PYARROW_MAX_SCAN_ROWS", "250000000")),
    "PYARROW_BATCH_ROWS": int(_ps3v21_env("PS3_PYARROW_BATCH_ROWS", "1000000")),
    "MAX_SPINE_EVENT_ROWS": int(_ps3v21_env("PS3_MAX_SPINE_EVENT_ROWS", "40000000")),

    # Overflow. "recent_window" keeps the newest episodes that fit and stamps
    # the reduced scope on every row; "defer" reproduces V21, which published
    # no dashboard table at all once the cap was crossed.
    "EPISODE_OVERFLOW_POLICY": _ps3v21_env("PS3_EPISODE_OVERFLOW_POLICY", "recent_window").lower(),

    # Modelling. Per-category first; a pooled model is only a labelled fallback
    # for a category that cannot meet its own support floor.
    "MODEL_PER_CATEGORY": _ps3v21_bool(_ps3v21_env("PS3_MODEL_PER_CATEGORY", "true"), True),
    "MIN_CATEGORY_TRAIN_ROWS": int(_ps3v21_env("PS3_MIN_CATEGORY_TRAIN_ROWS", "150")),

    # Bounded observational causal layer. Treatment is component attribution,
    # outcome is a repeat OOS episode within the horizon. Both come from the
    # spine, so this adds no source that could contradict PS1 or PS2.
    "ENABLE_CAUSAL_LAYER": _ps3v21_bool(_ps3v21_env("PS3_ENABLE_CAUSAL_LAYER", "true"), True),
    "CAUSAL_REPEAT_HORIZON_DAYS": int(_ps3v21_env("PS3_CAUSAL_REPEAT_HORIZON_DAYS", "30")),
    "CAUSAL_MIN_TREATED": int(_ps3v21_env("PS3_CAUSAL_MIN_TREATED", "150")),
    "CAUSAL_MIN_CONTROL": int(_ps3v21_env("PS3_CAUSAL_MIN_CONTROL", "150")),
    "CAUSAL_FOLDS": int(_ps3v21_env("PS3_CAUSAL_FOLDS", "4")),

    # Dashboard push. V21 emitted no trigger, so nothing downstream could know
    # a run had finished. run_complete.json is the loader's signal.
    "EMIT_RUN_COMPLETE": _ps3v21_bool(_ps3v21_env("PS3_EMIT_RUN_COMPLETE", "true"), True),
    "SMOKE_TEST_EPISODES": int(_ps3v21_env("PS3_SMOKE_TEST_EPISODES", "4000")),

    # V24: the source export is 76 columns wide. CLEAR_DTM carries sentinel
    # dates about 974 years out, so every [event, clear] interval is clamped and
    # the clamp is counted rather than silently applied.
    "MAX_EVENT_OPEN_MINUTES": float(_ps3v21_env("PS3_MAX_EVENT_OPEN_MINUTES", "10080")),
    # Severity observed in the source event record is an OBSERVATION, not an
    # invention, so it may be used as a label. It is refused when the profile
    # shows it is fully determined by EVENT_TYPE_ID, which would make it a
    # lookup on the event type rather than a fact about the incident.
    "USE_SOURCE_SEVERITY": _ps3v21_bool(_ps3v21_env("PS3_USE_SOURCE_SEVERITY", "true"), True),
    "PS3_REVISION": "source_first_ps1_ps2_ps3_label_aligned_v26",
}

# --- V22: dispatcher isolation and smoke-test safety ------------------------
if PS3V21_CONFIG["RUN_MODE"] not in {"PRODUCTION", "REPLAY"}:
    raise ValueError("PS3_RUN_MODE must be PRODUCTION or REPLAY.")
if PS3V21_CONFIG["EPISODE_OVERFLOW_POLICY"] not in {"recent_window", "defer"}:
    raise ValueError("PS3_EPISODE_OVERFLOW_POLICY must be recent_window or defer.")

PS3V21_PRODUCTION_TAIL = "ps3_outputs"
if PS3V21_CONFIG["RUN_MODE"] != "PRODUCTION":
    _root, _, _tail = PS3V21_CONFIG["S3_OUTPUT_PREFIX"].rpartition("/")
    if _tail == PS3V21_PRODUCTION_TAIL:
        PS3V21_CONFIG["S3_OUTPUT_PREFIX"] = f"{_root}/ps3_replay_outputs"
        PS3V21_CONFIG["EMIT_RUN_COMPLETE"] = False

# Synthetic data must never reach S3 under any configuration.
if PS3V21_CONFIG["SMOKE_TEST_SYNTHETIC"]:
    PS3V21_CONFIG["PUBLISH_TO_S3"] = False
    PS3V21_CONFIG["EMIT_RUN_COMPLETE"] = False
    PS3V21_CONFIG["WRITE_BOUNDED_SOURCE_EXPORT"] = False

PS3V21_STATE = {"run_id": None, "stage_audit": [], "publication": [], "models": {}}
print(json.dumps({k: v for k, v in PS3V21_CONFIG.items() if "EXPORT" not in k or bool(v)}, indent=2, default=str))


{
  "DATA_AS_OF_DATE": "2026-04-11",
  "LOOKBACK_DAYS": 730,
  "PS1_HORIZON_DAYS": 3,
  "PS1_OOS_EPISODE_GAP_DAYS": 3,
  "PS1_CURRENT_DEVICE_FILTER": true,
  "EXCLUDE_COMMANDED_OOS": false,
  "SOURCE_MODE": "auto",
  "ALLOW_DIRECT_SILVER_BOOTSTRAP": true,
  "RAW_SILVER_DEVICE_EVENT_PATH": "s3://cubic-mars-pm-s3-datalake-dev-gold-170202974600/chicago/silver/device_event_enriched/",
  "RAW_SILVER_DIM_DEVICE_PATH": "s3://cubic-mars-pm-s3-datalake-dev-gold-170202974600/chicago/silver/dim_device/",
  "EVIDENCE_LINK_TOLERANCE_HOURS": 72,
  "MAX_LOCAL_EPISODES": 2000000,
  "MAX_OPTIONAL_EXPORT_ROWS": 300000,
  "MAX_SHAP_EPISODES": 1000,
  "MIN_TRAIN_ROWS": 120,
  "MIN_TRAIN_CLASS_ROWS": 12,
  "MIN_VALID_CLASS_ROWS": 4,
  "MIN_MACRO_F1": 0.45,
  "MIN_MACRO_F1_LIFT": 0.05,
  "MIN_BALANCED_ACCURACY": 0.4,
  "PUBLISH_TO_S3": true,
  "S3_OUTPUT_PREFIX": "s3://cubic-mars-pm-s3-datalake-dev-artifacts-170202974600/chicago/ps3_outputs",
  "S3_BOOTSTRAP_PREFIX": "s3://cubic-mars-pm-s3-datalake-dev-arti

In [3]:
# 2. Safe dependencies, shared helpers, and run controls.
import hashlib
import math
import shutil
from io import BytesIO
from urllib.parse import urlparse

try:
    import numpy as np
    import pandas as pd
    PS3V21_PANDAS_READY = True
except Exception as exc:
    PS3V21_PANDAS_READY = False
    PS3V21_IMPORT_ERROR = f"pandas/numpy unavailable: {type(exc).__name__}: {exc}"

try:
    import boto3
    PS3V21_BOTO3_READY = True
except Exception:
    boto3 = None
    PS3V21_BOTO3_READY = False

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
    PS3V21_PYARROW_READY = True
except Exception:
    pa = pq = None
    PS3V21_PYARROW_READY = False

try:
    from sklearn.compose import ColumnTransformer
    from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
    from sklearn.impute import SimpleImputer
    from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, matthews_corrcoef
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder
    PS3V21_SKLEARN_READY = True
except Exception as exc:
    PS3V21_SKLEARN_READY = False
    PS3V21_SKLEARN_ERROR = f"scikit-learn unavailable: {type(exc).__name__}: {exc}"


def ps3v21_now_utc():
    return dt.datetime.now(dt.timezone.utc)


def ps3v21_run_id():
    """Sortable AND acceptable to the RDS loader's partition regex.

    V21 produced ids like ps3_20260802T135023Z_ad846ca0. They sort correctly,
    but the loader matches run_id=([0-9a-f-]+), and p, s, T and Z are outside
    that class, so every PS3 partition would have been skipped silently. Hex
    epoch seconds in the first eight characters of a valid UUID keeps
    chronological ordering and stays inside the accepted characters.
    """
    stamp = int(ps3v21_now_utc().timestamp())
    return str(uuid.UUID(format(stamp, "08x") + uuid.uuid4().hex[8:]))


def ps3v21_text(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass
    return str(value).strip()


def ps3v21_norm(value):
    return re.sub(r"[^a-z0-9]+", "", ps3v21_text(value).lower())


def ps3v21_category(series):
    values = series.astype("string").str.strip().str.upper()
    return values.replace({"GATES": "GATE", "TVMS": "TVM", "VALIDATORS": "VALIDATOR", "BMV": "VALIDATOR"})


def ps3v21_safe_datetime(series):
    # Explicitly coerces sentinels such as 3000-01-01 to NaT before any pandas
    # arithmetic. This prevents the previous OutOfBoundsDatetime failure.
    values = pd.to_datetime(series.astype("string"), errors="coerce", utc=True)
    try:
        values = values.dt.tz_convert(None)
    except Exception:
        pass
    lower = pd.Timestamp("1678-09-21")
    upper = pd.Timestamp("2262-04-11")
    return values.where(values.between(lower, upper), pd.NaT)


def ps3v21_safe_bool(series):
    return series.astype("string").str.strip().str.upper().isin(["TRUE", "T", "1", "YES", "Y"])


def ps3v21_stage(name, status="started", detail=None, **values):
    row = {"stage": name, "status": status, "detail": detail, "at_utc": ps3v21_now_utc().isoformat()}
    row.update(values)
    PS3V21_STATE.setdefault("stage_audit", []).append(row)
    return row


def ps3v21_error_detail(exc):
    return f"{type(exc).__name__}: {str(exc)[:1000]}"


def ps3v21_path_is_raw_silver(path):
    value = ps3v21_text(path).rstrip("/").lower()
    return value.endswith("/silver/device_event_enriched") or value.endswith("/silver/device_event_enriched/")


def ps3v21_s3a(uri):
    value = ps3v21_text(uri)
    return "s3a://" + value[5:] if value.startswith("s3://") else value


def ps3v21_parse_s3(uri):
    parsed = urlparse(ps3v21_text(uri))
    if parsed.scheme != "s3" or not parsed.netloc:
        raise ValueError("Expected an s3://bucket/key path")
    return parsed.netloc, parsed.path.lstrip("/")


def ps3v21_output_path(table_name, run_id=None):
    rid = run_id or PS3V21_STATE.get("run_id") or "uninitialised"
    return (
        f"{PS3V21_CONFIG['S3_OUTPUT_PREFIX']}/{table_name}/"
        f"computed_date={PS3V21_CONFIG['DATA_AS_OF_DATE']}/run_id={rid}"
    )


def ps3v21_dependency_frame():
    rows = [
        {"check": "pandas_numpy", "status": "passed" if PS3V21_PANDAS_READY else "failed", "detail": None if PS3V21_PANDAS_READY else globals().get("PS3V21_IMPORT_ERROR")},
        {"check": "scikit_learn", "status": "passed" if PS3V21_SKLEARN_READY else "warning", "detail": None if PS3V21_SKLEARN_READY else globals().get("PS3V21_SKLEARN_ERROR", "model lane will be skipped")},
        {"check": "boto3", "status": "passed" if PS3V21_BOTO3_READY else "warning", "detail": "Pandas Parquet publisher unavailable; Spark publisher remains usable." if not PS3V21_BOTO3_READY else None},
        {"check": "pyarrow", "status": "passed" if PS3V21_PYARROW_READY else "warning", "detail": "Pandas Parquet publisher unavailable; Spark publisher remains usable." if not PS3V21_PYARROW_READY else None},
    ]
    return pd.DataFrame(rows)


In [4]:
# 3. PS1-compatible OOS source acquisition. Gold is intentionally absent here.

def ps3v21_find_java():
    candidates = []
    current = os.environ.get("JAVA_HOME")
    if current:
        candidates.append(Path(current) / "bin" / "java")
    candidates.extend([
        Path("/opt/conda/lib/jvm/bin/java"),
        Path("/usr/lib/jvm/default-java/bin/java"),
        Path("/usr/bin/java"),
    ])
    on_path = shutil.which("java")
    if on_path:
        candidates.append(Path(on_path))
    for candidate in candidates:
        if candidate.exists() and os.access(candidate, os.X_OK):
            return candidate
    return None


def ps3v21_get_spark():
    existing = globals().get("spark")
    if existing is not None:
        try:
            existing.range(1).count()
            return existing, {"status": "passed", "mode": "existing", "detail": "Using the active governed Spark session."}
        except Exception:
            pass
    java = ps3v21_find_java()
    if java is None:
        return None, {"status": "not_ready", "mode": "none", "detail": "Java is unavailable. Choose a SageMaker image/kernel with Java and PySpark, or provide a bounded PS1 event/episode export."}
    os.environ.setdefault("JAVA_HOME", str(java.parent.parent))
    try:
        from pyspark.sql import SparkSession
        from pyspark import __version__ as spark_version
    except Exception as exc:
        return None, {"status": "not_ready", "mode": "none", "detail": "PySpark is unavailable: " + ps3v21_error_detail(exc)}
    try:
        # Existing managed Spark sessions are preferred. This local fallback is
        # only for Studio kernels and uses the same Hadoop AWS line that worked
        # in the PS1 notebooks.
        builder = (
            SparkSession.builder.appName("PS3_Source_First_V21")
            .master("local[*]")
            .config("spark.sql.session.timeZone", "UTC")
            .config("spark.sql.execution.arrow.pyspark.enabled", "true")
            .config("spark.driver.maxResultSize", "2g")
        )
        # Setting spark.jars.packages makes Spark run an Ivy resolve against Maven
        # Central on EVERY session start. That is fine in Studio and a liability in a
        # scheduled job, so the Processing image bakes the jars into $SPARK_HOME/jars
        # and sets PS3_SPARK_JARS_PACKAGES="" to skip the resolve. Unset (Studio) keeps
        # the original behaviour exactly.
        _pkgs = os.environ.get("PS3_SPARK_JARS_PACKAGES",
                               "org.apache.hadoop:hadoop-aws:3.3.4").strip()
        if _pkgs:
            builder = builder.config("spark.jars.packages", _pkgs)
        created = builder.getOrCreate()
        created.sparkContext.setLogLevel("WARN")
        created.range(1).count()
        globals()["spark"] = created
        return created, {"status": "passed", "mode": "local_ps1_compatible", "detail": f"Created Spark {spark_version} with JAVA_HOME={os.environ.get('JAVA_HOME')}."}
    except Exception as exc:
        return None, {"status": "not_ready", "mode": "none", "detail": "Spark could not start: " + ps3v21_error_detail(exc)}


def ps3v21_spark_pick(frame, candidates):
    lookup = {ps3v21_norm(column): column for column in frame.columns}
    for candidate in candidates:
        found = lookup.get(ps3v21_norm(candidate))
        if found:
            return found
    return None


def ps3v21_spark_truth(column):
    from pyspark.sql import functions as F
    return F.upper(F.trim(F.col(column).cast("string"))).isin("TRUE", "T", "1", "YES", "Y")


def ps3v21_read_spark_parquet(spark_session, path, source_name):
    try:
        frame = spark_session.read.parquet(ps3v21_s3a(path))
        return frame, {"source": source_name, "status": "loaded", "reference": path, "detail": None}
    except Exception as exc:
        return None, {"source": source_name, "status": "unavailable", "reference": path, "detail": ps3v21_error_detail(exc)}


def ps3v21_normalise_direct_events(spark_session, raw, source_name, apply_current_device_filter=False):
    """Return a direct-Silver PS1 hardware-OOS SET event fact DataFrame."""
    from pyspark.sql import functions as F

    device_key = ps3v21_spark_pick(raw, ["device_key", "device_sk"])
    device_id = ps3v21_spark_pick(raw, ["device_id", "device", "serial_number", "asset_id"])
    category = ps3v21_spark_pick(raw, ["mars_device_category", "device_category", "device_type"])
    event_time = ps3v21_spark_pick(raw, ["event_dtm", "event_timestamp", "event_datetime", "oos_start_timestamp"])
    hardware = ps3v21_spark_pick(raw, ["is_hardware_oos_event", "hardware_oos_flag", "is_hardware_oos"])
    state = ps3v21_spark_pick(raw, ["event_state_type_name", "event_state", "state_type", "oos_state"])
    event_id = ps3v21_spark_pick(raw, ["dw_device_event_id", "device_event_id", "event_id", "oos_event_id", "source_event_id"])
    commanded = ps3v21_spark_pick(raw, ["is_commanded_oos_event", "is_commanded_oos", "commanded_oos_flag"])
    component = ps3v21_spark_pick(raw, ["component_type", "device_component", "component", "failure_component", "subsystem"])
    # V22: carried through for facility rollups, bus context and serial-level
    # reliability. These widen the projection only; they do not touch the OOS test.
    facility = ps3v21_spark_pick(raw, ["facility_id", "facility"])
    bus = ps3v21_spark_pick(raw, ["bus_id", "vehicle_id"])
    serial = ps3v21_spark_pick(raw, ["component_serial_nbr", "component_serial", "serial_nbr"])
    missing = [logical for logical, actual in {
        "device_id": device_id,
        "mars_device_category": category,
        "event timestamp": event_time,
        "is_hardware_oos_event": hardware,
        "EVENT_STATE_TYPE_NAME": state,
    }.items() if actual is None]
    if missing:
        return None, {"source": source_name, "status": "missing_required_columns", "detail": "Missing PS1 hardware-OOS SET fields: " + ", ".join(missing), "selected_rows": 0}

    timestamp = F.to_timestamp(F.col(event_time).cast("string"))
    category_expr = F.upper(F.trim(F.col(category).cast("string")))
    category_expr = F.when(category_expr == "GATES", F.lit("GATE")).when(category_expr == "TVMS", F.lit("TVM")).when(category_expr.isin("VALIDATORS", "BMV"), F.lit("VALIDATOR")).otherwise(category_expr)
    start_date = (pd.Timestamp(PS3V21_CONFIG["DATA_AS_OF_DATE"]) - pd.Timedelta(days=PS3V21_CONFIG["LOOKBACK_DAYS"])).date().isoformat()
    end_date = PS3V21_CONFIG["DATA_AS_OF_DATE"]
    filtered = raw.where(
        ps3v21_spark_truth(hardware)
        & F.upper(F.trim(F.col(state).cast("string"))).eqNullSafe("SET")
        & category_expr.isin("TVM", "GATE", "VALIDATOR")
        & F.col(device_id).isNotNull()
        & timestamp.isNotNull()
        & F.to_date(timestamp).between(start_date, end_date)
        # Explicitly protects pandas conversion from 3000-01-01 and similar
        # warehouse sentinel dates.
        & (timestamp >= F.lit("1678-09-21 00:00:00").cast("timestamp"))
        & (timestamp <= F.lit("2262-04-11 00:00:00").cast("timestamp"))
    )
    source = filtered.select(
        F.col(device_key).cast("string").alias("device_key") if device_key else F.lit(None).cast("string").alias("device_key"),
        F.col(device_id).cast("string").alias("device_id"),
        category_expr.alias("mars_device_category"),
        timestamp.alias("event_timestamp"),
        (F.col(event_id).cast("string") if event_id else F.lit(None).cast("string")).alias("oos_event_id"),
        (ps3v21_spark_truth(commanded) if commanded else F.lit(False)).alias("is_commanded_oos_event"),
        (F.col(component).cast("string") if component else F.lit(None).cast("string")).alias("observed_event_component"),
        (F.col(facility).cast("string") if facility else F.lit(None).cast("string")).alias("facility_id"),
        (F.col(bus).cast("string") if bus else F.lit(None).cast("string")).alias("bus_id"),
        (F.col(serial).cast("string") if serial else F.lit(None).cast("string")).alias("component_serial_nbr"),
    )
    if PS3V21_CONFIG["EXCLUDE_COMMANDED_OOS"]:
        source = source.where(~F.col("is_commanded_oos_event"))
    source = source.withColumn(
        "oos_event_id",
        F.coalesce(
            F.when(F.length(F.trim(F.col("oos_event_id"))) > 0, F.trim(F.col("oos_event_id"))),
            F.sha2(F.concat_ws("|", F.col("device_id"), F.col("mars_device_category"), F.col("event_timestamp").cast("string")), 256),
        ),
    ).dropDuplicates(["oos_event_id", "device_id", "mars_device_category", "event_timestamp"])

    current_filter_status = "not_requested"
    if apply_current_device_filter and PS3V21_CONFIG["PS1_CURRENT_DEVICE_FILTER"]:
        dim, dim_status = ps3v21_read_spark_parquet(spark_session, PS3V21_CONFIG["RAW_SILVER_DIM_DEVICE_PATH"], "silver_dim_device")
        dim_key = ps3v21_spark_pick(dim, ["device_key", "device_sk"]) if dim is not None else None
        dim_current = ps3v21_spark_pick(dim, ["is_current", "current_flag"]) if dim is not None else None
        if dim is not None and device_key and dim_key and dim_current:
            current_devices = dim.where(ps3v21_spark_truth(dim_current)).select(F.col(dim_key).cast("string").alias("device_key")).distinct()
            source = source.join(F.broadcast(current_devices), on="device_key", how="inner")
            current_filter_status = "applied"
        else:
            current_filter_status = "not_applied_dim_unavailable_or_schema_missing"
    details = {
        "source": source_name,
        "status": "loaded",
        "detail": "PS1-compatible: hardware OOS=true AND state=SET; chargeability not read or used.",
        "current_device_filter": current_filter_status,
        "window_start": start_date,
        "window_end": end_date,
        "selected_rows": None,
    }
    return source, details


def ps3v21_sessionise_events(event_facts):
    from pyspark.sql import functions as F
    from pyspark.sql.window import Window
    ordered = Window.partitionBy("mars_device_category", "device_id").orderBy("event_timestamp", "oos_event_id")
    cumulative = ordered.rowsBetween(Window.unboundedPreceding, Window.currentRow)
    gap_seconds = int(PS3V21_CONFIG["PS1_OOS_EPISODE_GAP_DAYS"]) * 86400
    staged = event_facts.withColumn("_prior_event_timestamp", F.lag("event_timestamp").over(ordered))
    staged = staged.withColumn(
        "_new_episode",
        F.when(F.col("_prior_event_timestamp").isNull(), F.lit(1)).when(
            F.col("event_timestamp").cast("long") - F.col("_prior_event_timestamp").cast("long") > gap_seconds,
            F.lit(1),
        ).otherwise(F.lit(0)),
    ).withColumn("_episode_number", F.sum("_new_episode").over(cumulative))
    staged = staged.withColumn(
        "oos_episode_id",
        F.sha2(F.concat_ws("|", F.col("mars_device_category"), F.col("device_id"), F.col("_episode_number").cast("string")), 256),
    )
    episodes = staged.groupBy("oos_episode_id", "device_id", "mars_device_category").agg(
        F.min("event_timestamp").alias("episode_start"),
        F.max("event_timestamp").alias("episode_last_signal"),
        F.count("*").cast("long").alias("oos_set_event_count"),
        F.first("oos_event_id", ignorenulls=True).alias("first_oos_event_id"),
        F.first("observed_event_component", ignorenulls=True).alias("observed_event_component"),
        F.max(F.col("is_commanded_oos_event").cast("int")).cast("boolean").alias("contains_commanded_oos_signal"),
        F.first("facility_id", ignorenulls=True).alias("facility_id"),
        F.first("bus_id", ignorenulls=True).alias("bus_id"),
        F.first("component_serial_nbr", ignorenulls=True).alias("component_serial_nbr"),
    ).withColumn(
        "set_signal_span_minutes",
        (F.col("episode_last_signal").cast("long") - F.col("episode_start").cast("long")) / F.lit(60.0),
    ).withColumn(
        "oos_fact_definition",
        F.lit(f"PS1 hardware-OOS SET session; a new session starts after > {PS3V21_CONFIG['PS1_OOS_EPISODE_GAP_DAYS']} days"),
    )
    return episodes


def ps3v21_read_precomputed_episodes(spark_session, path):
    raw, read_status = ps3v21_read_spark_parquet(spark_session, path, "verified_ps1_oos_episode_export")
    if raw is None:
        return None, read_status
    from pyspark.sql import functions as F
    device = ps3v21_spark_pick(raw, ["device_id", "device", "serial_number"])
    category = ps3v21_spark_pick(raw, ["mars_device_category", "device_category", "device_type"])
    start = ps3v21_spark_pick(raw, ["episode_start", "oos_episode_start", "event_timestamp", "event_dtm", "start_dtm"])
    end = ps3v21_spark_pick(raw, ["episode_last_signal", "episode_end", "oos_episode_end", "event_end_timestamp", "clear_dtm"])
    episode = ps3v21_spark_pick(raw, ["oos_episode_id", "episode_id", "hardware_oos_episode_id"])
    event = ps3v21_spark_pick(raw, ["oos_event_id", "event_id", "device_event_id"])
    component = ps3v21_spark_pick(raw, ["component_type", "device_component", "component", "failure_component"])
    missing = [name for name, value in {"device_id": device, "mars_device_category": category, "episode_start": start}.items() if value is None]
    if missing:
        return None, {"source": "verified_ps1_oos_episode_export", "status": "missing_required_columns", "reference": path, "detail": ", ".join(missing)}
    semantic_columns = [c for c in raw.columns if ps3v21_norm(c) in {"sourcesemantics", "targetdefinition", "labeldefinition", "oosdefinition", "episodedefinition"}]
    semantic_text = ""
    if semantic_columns:
        try:
            semantic_text = " ".join(str(v) for v in raw.select(*semantic_columns).limit(5).toPandas().fillna("").astype(str).values.ravel()).lower()
        except Exception:
            semantic_text = ""
    raw_hw = ps3v21_spark_pick(raw, ["is_hardware_oos_event", "hardware_oos_flag"])
    raw_state = ps3v21_spark_pick(raw, ["event_state_type_name", "event_state", "state_type"])
    has_raw_proof = raw_hw is not None and raw_state is not None
    has_semantic_proof = "hardware" in semantic_text and ("oos" in semantic_text or "out of service" in semantic_text) and ("set" in semantic_text or "episode" in semantic_text)
    if not (has_raw_proof or has_semantic_proof):
        return None, {"source": "verified_ps1_oos_episode_export", "status": "unverified_episode_semantics", "reference": path, "detail": "Need explicit hardware-OOS SET fields or source/target definition proving hardware OOS SET/episode semantics."}
    category_expr = F.upper(F.trim(F.col(category).cast("string")))
    category_expr = F.when(category_expr == "GATES", F.lit("GATE")).when(category_expr == "TVMS", F.lit("TVM")).when(category_expr.isin("VALIDATORS", "BMV"), F.lit("VALIDATOR")).otherwise(category_expr)
    start_ts = F.to_timestamp(F.col(start).cast("string"))
    end_ts = F.to_timestamp(F.col(end).cast("string")) if end else start_ts
    facts = raw.select(
        F.col(device).cast("string").alias("device_id"),
        category_expr.alias("mars_device_category"),
        start_ts.alias("episode_start"),
        F.coalesce(end_ts, start_ts).alias("episode_last_signal"),
        (F.col(episode).cast("string") if episode else F.lit(None).cast("string")).alias("oos_episode_id"),
        (F.col(event).cast("string") if event else F.lit(None).cast("string")).alias("first_oos_event_id"),
        (F.col(component).cast("string") if component else F.lit(None).cast("string")).alias("observed_event_component"),
    ).where(
        F.col("device_id").isNotNull()
        & F.col("mars_device_category").isin("TVM", "GATE", "VALIDATOR")
        & F.col("episode_start").isNotNull()
        & (F.col("episode_start") >= F.lit("1678-09-21 00:00:00").cast("timestamp"))
        & (F.col("episode_start") <= F.lit("2262-04-11 00:00:00").cast("timestamp"))
    ).withColumn(
        "oos_episode_id",
        F.coalesce(F.when(F.length(F.trim(F.col("oos_episode_id"))) > 0, F.trim(F.col("oos_episode_id"))), F.sha2(F.concat_ws("|", F.col("device_id"), F.col("mars_device_category"), F.col("episode_start").cast("string")), 256)),
    ).withColumn("oos_set_event_count", F.lit(1).cast("long")).withColumn("set_signal_span_minutes", F.lit(0.0)).withColumn("contains_commanded_oos_signal", F.lit(False)).withColumn(
        "oos_fact_definition", F.lit("Verified PS1-compatible hardware-OOS episode export")
    ).dropDuplicates(["oos_episode_id", "device_id", "mars_device_category", "episode_start"])
    return facts, {"source": "verified_ps1_oos_episode_export", "status": "loaded", "reference": path, "detail": "Verified PS1 hardware-OOS SET/episode semantics."}


def ps3v21_synthetic_episodes():
    """Synthetic smoke-test spine. Never publishes.

    V21's synthetic frame had four rows, which is below every model support
    floor, so the smoke test could only ever prove that the notebook started.
    This one generates a realistic multi-thousand-episode spine with component
    attribution and a severity signal so the feature, model, causal and table
    lanes are all genuinely exercised. Publication is forced off in this mode.
    """
    rng = np.random.default_rng(20260802)
    n_devices = 240
    target = int(PS3V21_CONFIG["SMOKE_TEST_EPISODES"])
    categories = np.array(["TVM", "GATE", "VALIDATOR"])
    devices = [f"{categories[i % 3][:3]}{i:05d}" for i in range(n_devices)]
    device_category = {d: categories[i % 3] for i, d in enumerate(devices)}
    facilities = {d: f"{(i % 40) + 1}" for i, d in enumerate(devices)}
    components = np.array(["CSC_READER", "PRINTER", "GATE_MECH", "BANKCARD", "COMMS", "DOPP"])
    as_of = pd.Timestamp(PS3V21_CONFIG["DATA_AS_OF_DATE"])
    start = as_of - pd.Timedelta(days=min(PS3V21_CONFIG["LOOKBACK_DAYS"], 720))
    chosen = rng.choice(devices, target)
    offsets = rng.integers(0, int((as_of - start).days) * 86400, target)
    starts = pd.to_datetime([start + pd.Timedelta(seconds=int(o)) for o in offsets])
    component = rng.choice(components, target, p=[0.28, 0.14, 0.2, 0.12, 0.17, 0.09])
    # A deliberate, known signal so the gates have something real to detect.
    severity_p = np.where(np.isin(component, ["GATE_MECH", "COMMS"]), 0.55, 0.2)
    severity = np.where(rng.random(target) < severity_p, "Critical", "Minor")
    frame = pd.DataFrame({
        "device_id": pd.Series(chosen, dtype="string"),
        "mars_device_category": pd.Series([device_category[d] for d in chosen], dtype="string"),
        "episode_start": starts,
        "oos_set_event_count": rng.integers(1, 12, target).astype("int64"),
        "observed_event_component": pd.Series(component, dtype="string"),
        "facility_id": pd.Series([facilities[d] for d in chosen], dtype="string"),
        "bus_id": pd.Series(pd.NA, index=range(target), dtype="string"),
        "component_serial_nbr": pd.Series([f"0x{abs(hash(d)) % (16 ** 12):012X}" for d in chosen], dtype="string"),
        "contains_commanded_oos_signal": rng.random(target) < 0.12,
        "_smoke_severity": pd.Series(severity, dtype="string"),
    })
    frame["episode_last_signal"] = frame["episode_start"] + pd.to_timedelta(rng.integers(0, 240, target), unit="m")
    frame["set_signal_span_minutes"] = (frame["episode_last_signal"] - frame["episode_start"]).dt.total_seconds() / 60.0
    frame["oos_episode_id"] = ps3v22_sha256_series(
        frame.assign(_s=frame["episode_start"].astype("string")), ["mars_device_category", "device_id", "_s"])
    frame["first_oos_event_id"] = frame["oos_episode_id"]
    frame["oos_fact_definition"] = "SYNTHETIC SMOKE TEST - NOT FOR DELIVERY"
    return frame.sort_values("episode_start").reset_index(drop=True)


def ps3v21_load_oos_spine():
    """Resolve one source path without any Gold dependency or unhandled exception.

    V22 route order: synthetic smoke test -> JVM-free pyarrow spine -> the
    original Spark routes. The OOS definition is identical on every route:
    is_hardware_oos_event = true AND EVENT_STATE_TYPE_NAME = SET, restricted to
    TVM/GATE/VALIDATOR, current devices only, sessionised on the same gap rule.
    Only the engine changes.
    """
    audit = []
    if PS3V21_CONFIG["SMOKE_TEST_SYNTHETIC"]:
        demo = ps3v21_synthetic_episodes()
        audit.append({"source": "synthetic_smoke_test", "status": "loaded", "rows": len(demo),
                      "detail": "SYNTHETIC DATA. Publication is disabled in this mode."})
        return {"status": "loaded", "mode": "synthetic_smoke_test", "episodes_pd": demo,
                "episodes_spark": None, "event_facts_spark": None, "audit": audit}

    # --- V22: JVM-free route, tried first ---------------------------------
    if PS3V21_CONFIG["PREFER_PYARROW_SPINE"] and PS3V21_CONFIG["ALLOW_DIRECT_SILVER_BOOTSTRAP"]:
        events, arrow_status = ps3v22_read_arrow_spine(
            PS3V21_CONFIG["RAW_SILVER_DEVICE_EVENT_PATH"], "arrow_direct_silver_bootstrap", True)
        audit.append(arrow_status)
        if events is not None and len(events) > 0:
            episodes = ps3v22_sessionise_events_pandas(events)
            return {"status": "loaded", "mode": "arrow_direct_silver_bootstrap", "episodes_pd": episodes,
                    "episodes_spark": None, "event_facts_spark": None, "audit": audit,
                    "event_facts_pd": events}
        if events is not None and len(events) == 0:
            audit[-1] = {**arrow_status, "status": "loaded_but_empty",
                         "detail": "The filter matched no rows. Check DATA_AS_OF_DATE and LOOKBACK_DAYS."}

    spark_session, spark_status = ps3v21_get_spark()
    audit.append({"source": "spark_runtime", **spark_status})
    if spark_session is None:
        return {"status": "not_ready", "mode": "no_spark", "episodes_pd": pd.DataFrame(), "episodes_spark": None,
                "event_facts_spark": None, "audit": audit,
                "detail": "The JVM-free route did not yield rows and Spark is unavailable. " + spark_status["detail"]}

    mode = PS3V21_CONFIG["SOURCE_MODE"]
    attempts = []
    direct_path = PS3V21_CONFIG["PS1_DIRECT_EVENT_EXPORT"]
    episode_path = PS3V21_CONFIG["PS1_VERIFIED_EPISODE_EXPORT"]
    if mode in {"auto", "direct_export"} and direct_path:
        if ps3v21_path_is_raw_silver(direct_path):
            attempts.append({"source": "direct_event_export", "status": "rejected", "reference": direct_path, "detail": "Raw Silver was supplied as an export. Use bootstrap mode or a bounded Parquet export path."})
        else:
            raw, raw_status = ps3v21_read_spark_parquet(spark_session, direct_path, "ps1_direct_event_export")
            if raw is not None:
                events, event_status = ps3v21_normalise_direct_events(spark_session, raw, "ps1_direct_event_export", False)
                attempts.append({"source": "direct_event_export", **event_status, "reference": direct_path})
                if events is not None and events.limit(1).count() > 0:
                    episodes = ps3v21_sessionise_events(events)
                    return {"status": "loaded", "mode": "bounded_direct_event_export", "episodes_pd": None, "episodes_spark": episodes, "event_facts_spark": events, "audit": audit + attempts}
            else:
                attempts.append(raw_status)
    if mode in {"auto", "episode_export"} and episode_path:
        episodes, episode_status = ps3v21_read_precomputed_episodes(spark_session, episode_path)
        attempts.append(episode_status)
        if episodes is not None and episodes.limit(1).count() > 0:
            return {"status": "loaded", "mode": "verified_episode_export", "episodes_pd": None, "episodes_spark": episodes, "event_facts_spark": None, "audit": audit + attempts}
    if mode in {"auto", "bootstrap_silver"} and PS3V21_CONFIG["ALLOW_DIRECT_SILVER_BOOTSTRAP"]:
        raw, raw_status = ps3v21_read_spark_parquet(spark_session, PS3V21_CONFIG["RAW_SILVER_DEVICE_EVENT_PATH"], "silver_device_event_enriched")
        if raw is not None:
            events, event_status = ps3v21_normalise_direct_events(spark_session, raw, "silver_device_event_enriched", True)
            attempts.append({"source": "direct_silver_bootstrap", **event_status, "reference": PS3V21_CONFIG["RAW_SILVER_DEVICE_EVENT_PATH"]})
            if events is not None and events.limit(1).count() > 0:
                episodes = ps3v21_sessionise_events(events)
                return {"status": "loaded", "mode": "direct_silver_bootstrap", "episodes_pd": None, "episodes_spark": episodes, "event_facts_spark": events, "audit": audit + attempts}
        else:
            attempts.append(raw_status)
    return {"status": "not_ready", "mode": "no_verified_oos_source", "episodes_pd": pd.DataFrame(), "episodes_spark": None, "event_facts_spark": None, "audit": audit + attempts, "detail": "No usable PS1-compatible OOS source was available. The audit explains the exact source, Spark, schema, or permission issue."}


def ps3v21_collect_episodes(source):
    """Bring the episode spine into pandas.

    V22: when the spine exceeds the local limit, V21 returned NOTHING but a raw
    table - no labels, no models, no dashboard tables. That is a severe and
    almost invisible degradation. The default policy now keeps the most recent
    window that fits and stamps the reduced scope on every published row and on
    the run control, so the dashboard can never imply full coverage it does not
    have. Set PS3_EPISODE_OVERFLOW_POLICY=defer for the old behaviour.
    """
    frame = None
    if isinstance(source.get("episodes_pd"), pd.DataFrame) and not source["episodes_pd"].empty:
        frame = source["episodes_pd"].copy()
    elif source.get("episodes_spark") is not None:
        try:
            frame = source["episodes_spark"].toPandas()
        except Exception as exc:
            return pd.DataFrame(), {"status": "collect_failed", "rows": 0, "detail": ps3v21_error_detail(exc)}
    if frame is None:
        return pd.DataFrame(), {"status": "not_ready", "rows": 0, "detail": source.get("detail")}

    for column in ["episode_start", "episode_last_signal"]:
        if column in frame:
            frame[column] = ps3v21_safe_datetime(frame[column])

    total = len(frame)
    limit = PS3V21_CONFIG["MAX_LOCAL_EPISODES"]
    if total <= limit:
        frame["episode_scope_status"] = "complete_lookback_window"
        frame["episode_scope_start"] = frame["episode_start"].min()
        return frame, {"status": "loaded", "rows": total, "detail": "Episode table collected in full without sampling."}

    if PS3V21_CONFIG["EPISODE_OVERFLOW_POLICY"] == "defer":
        return pd.DataFrame(), {"status": "too_large_for_local_enrichment", "rows": total,
                                "detail": f"{total:,} episodes exceed MAX_LOCAL_EPISODES={limit:,}; enrichment deferred without sampling."}

    kept = frame.sort_values("episode_start", ascending=False).head(limit).sort_values("episode_start").reset_index(drop=True)
    scope_start = kept["episode_start"].min()
    kept["episode_scope_status"] = "truncated_to_most_recent_window"
    kept["episode_scope_start"] = scope_start
    return kept, {"status": "loaded_truncated", "rows": len(kept), "total_rows": total,
                  "detail": (f"{total:,} episodes exceed MAX_LOCAL_EPISODES={limit:,}. Kept the most recent "
                             f"{len(kept):,} from {scope_start}. Every published row carries "
                             "episode_scope_status=truncated_to_most_recent_window.")}


def run_ps3_v21_preflight():
    if not PS3V21_PANDAS_READY:
        return pd.DataFrame([{"check": "pandas_numpy", "status": "failed", "detail": globals().get("PS3V21_IMPORT_ERROR")}])
    rows = ps3v21_dependency_frame().to_dict("records")
    # V22: do not start a JVM to report on a route we are not taking. V21
    # probed Spark here unconditionally and then reported ps1_oos_fact_source
    # as not_ready purely because Java was absent, even though the spine is now
    # readable without it.
    _arrow_route = bool(PS3V21_PYARROW_READY and PS3V21_CONFIG["PREFER_PYARROW_SPINE"] and PS3V21_CONFIG["ALLOW_DIRECT_SILVER_BOOTSTRAP"])
    if PS3V21_CONFIG["SMOKE_TEST_SYNTHETIC"]:
        spark_session, spark_status = None, {"status": "not_probed", "detail": "Synthetic smoke test does not use Spark."}
    elif _arrow_route:
        spark_session, spark_status = None, {"status": "not_probed", "detail": "The JVM-free pyarrow route is preferred. Spark is probed only if that route yields no rows."}
    else:
        spark_session, spark_status = ps3v21_get_spark()
    rows.append({"check": "spark_runtime", "status": spark_status["status"], "detail": spark_status["detail"], "reference": None})
    if PS3V21_CONFIG["SMOKE_TEST_SYNTHETIC"]:
        source_text = "synthetic smoke test"
    elif PS3V21_CONFIG["PS1_DIRECT_EVENT_EXPORT"]:
        source_text = "provided direct event export"
    elif PS3V21_CONFIG["PS1_VERIFIED_EPISODE_EXPORT"]:
        source_text = "provided episode export"
    elif _arrow_route:
        source_text = "JVM-free pyarrow direct Silver bootstrap, Spark fallback"
    else:
        source_text = "direct Silver bootstrap via Spark"
    source_status = "passed" if (
        PS3V21_CONFIG["SMOKE_TEST_SYNTHETIC"]
        or PS3V21_CONFIG["PS1_DIRECT_EVENT_EXPORT"]
        or PS3V21_CONFIG["PS1_VERIFIED_EPISODE_EXPORT"]
        or _arrow_route
        or (PS3V21_CONFIG["ALLOW_DIRECT_SILVER_BOOTSTRAP"] and spark_session is not None)
    ) else "not_ready"
    rows.append({"check": "ps1_oos_fact_source", "status": source_status, "detail": f"Source route: {source_text}. Gold is not used for OOS discovery.", "reference": PS3V21_CONFIG["PS1_DIRECT_EVENT_EXPORT"] or PS3V21_CONFIG["PS1_VERIFIED_EPISODE_EXPORT"] or PS3V21_CONFIG["RAW_SILVER_DEVICE_EVENT_PATH"]})
    rows.append({"check": "ps3_gold_label_lane", "status": "optional_configured" if PS3V21_CONFIG["GOLD_INCIDENT_LABEL_EXPORT"] else "optional_not_configured", "detail": "Optional label/evidence enrichment only; it is not an OOS-source blocker.", "reference": PS3V21_CONFIG["GOLD_INCIDENT_LABEL_EXPORT"] or None})
    _prefix_tail = PS3V21_CONFIG["S3_OUTPUT_PREFIX"].rpartition("/")[2]
    _prefix_ok = PS3V21_CONFIG["S3_OUTPUT_PREFIX"].startswith("s3://") and _prefix_tail in {"ps3_outputs", "ps3_replay_outputs"}
    rows.append({"check": "s3_output_prefix", "status": "passed" if _prefix_ok else "failed", "detail": ("Immutable table/run partition output on the artifacts bucket; no broad cleanup. " f"RUN_MODE={PS3V21_CONFIG['RUN_MODE']}, loader trigger " f"{'enabled' if PS3V21_CONFIG['EMIT_RUN_COMPLETE'] else 'disabled'}."), "reference": PS3V21_CONFIG["S3_OUTPUT_PREFIX"]})
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# V22 ADDITION: JVM-free spine. Reads the same PS1 direct-Silver export with
# pyarrow instead of Spark, applying byte-identical filter and episode
# semantics. Spark remains the fallback, so nothing is lost when it is present.
# ---------------------------------------------------------------------------
def ps3v22_sha256(*parts):
    """Reproduce Spark sha2(concat_ws('|', ...), 256). concat_ws skips nulls."""
    kept = [p for p in parts if p is not None and p == p and str(p) != "<NA>"]
    return hashlib.sha256("|".join(str(p) for p in kept).encode("utf-8")).hexdigest()


def ps3v22_sha256_series(frame, columns):
    """Vectorised equivalent of sha2(concat_ws('|', columns), 256)."""
    parts = []
    for column in columns:
        values = frame[column].astype("string")
        parts.append(values)
    joined = None
    for values in parts:
        piece = values.fillna("")
        mask = values.notna()
        if joined is None:
            joined = piece.where(mask, "")
            present = mask
        else:
            sep = np.where(present & mask, "|", "")
            joined = joined + sep + piece.where(mask, "")
            present = present | mask
    return pd.Series(
        [hashlib.sha256(v.encode("utf-8")).hexdigest() for v in joined.fillna("")],
        index=frame.index, dtype="string",
    )


def ps3v22_pick_arrow(names, candidates):
    lookup = {ps3v21_norm(n): n for n in names}
    for candidate in candidates:
        actual = lookup.get(ps3v21_norm(candidate))
        if actual is not None:
            return actual
    return None


def ps3v22_arrow_truth(column):
    """Match ps3v21_spark_truth: treat true/t/1/yes/y as true, everything else false."""
    if pd.api.types.is_bool_dtype(column):
        return column.fillna(False)
    if pd.api.types.is_numeric_dtype(column):
        return column.fillna(0) != 0
    return column.astype("string").str.strip().str.upper().isin(["TRUE", "T", "1", "YES", "Y"])


# ---------------------------------------------------------------------------
# V24: the source is 76 columns wide, not the narrow export assumed in V22/V23.
# These are the columns that were sitting there unused. Nothing here changes the
# OOS test -- it is still is_hardware_oos_event AND EVENT_STATE_TYPE_NAME=SET.
# ---------------------------------------------------------------------------
PS3V22_COLUMN_ALIASES = {
    "device_key":     ["device_key", "device_sk"],
    "device_id":      ["device_id", "device", "serial_number", "asset_id"],
    "category":       ["mars_device_category", "device_category", "device_type"],
    "event_time":     ["event_dtm", "event_timestamp", "event_datetime", "oos_start_timestamp"],
    "hardware":       ["is_hardware_oos_event", "hardware_oos_flag", "is_hardware_oos"],
    "state":          ["event_state_type_name", "event_state", "state_type", "oos_state"],
    "event_id":       ["dw_device_event_id", "device_event_id", "event_id", "oos_event_id", "source_event_id"],
    "commanded":      ["is_commanded_oos_event", "is_commanded_oos", "commanded_oos_flag"],
    # V24 FIX: the export calls this COMPONENT_TYPE_NAME. The V23 alias list
    # normalised to "componenttype" and matched nothing, so every episode came
    # back with a null component. That is why attribution, the repeat-interval
    # table and the whole causal layer were empty on a run that otherwise worked.
    "component":      ["component_type_name", "component_type", "device_component", "component",
                       "failure_component"],
    "subsystem":      ["component_subsystem", "subsystem"],
    "component_pos":  ["component_position"],
    "facility":       ["facility_id", "facility"],
    "facility_name":  ["facility_name"],
    "bus":            ["bus_id", "vehicle_id"],
    "serial":         ["component_serial_nbr", "component_serial", "serial_nbr"],
    "clear_time":     ["clear_dtm", "clear_datetime"],
    "auto_clear":     ["automatic_clear_flag", "auto_clear_flag"],
    "duration_clear": ["duration_to_clear_min", "duration_to_clear"],
    "event_type_id":  ["event_type_id"],
    "event_type":     ["event_type_name", "event_type_desc"],
    "severity":       ["severity"],
    "event_type_sev": ["event_type_severity"],
    "priority":       ["event_priority"],
    "service_call":   ["requires_service_call"],
}

# Text carried per episode: first non-null wins, matching the Spark first(ignorenulls) rule.
PS3V22_EPISODE_FIRST = [
    "observed_event_component", "component_subsystem", "component_position",
    "facility_id", "facility_name", "bus_id", "component_serial_nbr",
    "event_type_name", "event_type_id", "observed_event_severity",
    "event_type_severity", "event_priority",
]


def ps3v22_resolve_columns(names):
    """Resolve every logical slot against the actual schema, case- and
    punctuation-insensitively. One table, so an alias gap is visible in one place."""
    return {slot: ps3v22_pick_arrow(names, cands) for slot, cands in PS3V22_COLUMN_ALIASES.items()}


def ps3v22_episode_downtime(events):
    """Union of [event, clear] intervals per episode -- honest OOS minutes.

    Deliberately NOT sum(duration_to_clear_min). PS2's availability metric sums
    per-episode durations that can overlap on the same device, which is how it
    reached 22.9 out-of-service hours per validator per day. PS3 computes the
    union instead and publishes the naive sum beside it so the gap is visible.

    CLEAR_DTM in this source carries sentinel dates roughly 974 years out, so
    each interval is clamped to MAX_EVENT_OPEN_MINUTES and the clamp is counted,
    never silently applied.
    """
    if "clear_timestamp" not in events.columns:
        return pd.DataFrame(columns=["oos_episode_id", "oos_minutes_union", "oos_minutes_naive_sum",
                                     "events_with_clear", "events_clear_clamped"])
    cap = float(PS3V21_CONFIG["MAX_EVENT_OPEN_MINUTES"])
    frame = events[["oos_episode_id", "event_timestamp", "clear_timestamp"]].copy()
    frame["_end_raw"] = frame["clear_timestamp"]
    has_clear = frame["_end_raw"].notna() & (frame["_end_raw"] >= frame["event_timestamp"])
    frame = frame.loc[has_clear].copy()
    if frame.empty:
        return pd.DataFrame(columns=["oos_episode_id", "oos_minutes_union", "oos_minutes_naive_sum",
                                     "events_with_clear", "events_clear_clamped"])
    ceiling = frame["event_timestamp"] + pd.to_timedelta(cap, unit="m")
    frame["_clamped"] = frame["_end_raw"] > ceiling
    frame["_end"] = frame["_end_raw"].where(~frame["_clamped"], ceiling)
    frame["_naive_min"] = (frame["_end"] - frame["event_timestamp"]).dt.total_seconds() / 60.0

    frame = frame.sort_values(["oos_episode_id", "event_timestamp", "_end"], kind="mergesort")
    group = frame["oos_episode_id"]
    prior_max_end = frame.groupby(group, sort=False)["_end"].cummax().groupby(group, sort=False).shift(1)
    new_block = prior_max_end.isna() | (frame["event_timestamp"] > prior_max_end)
    frame["_block"] = new_block.groupby(group, sort=False).cumsum()
    blocks = (
        frame.groupby(["oos_episode_id", "_block"], sort=False)
        .agg(bs=("event_timestamp", "min"), be=("_end", "max")).reset_index()
    )
    blocks["_minutes"] = (blocks["be"] - blocks["bs"]).dt.total_seconds() / 60.0
    union = blocks.groupby("oos_episode_id", sort=False)["_minutes"].sum().rename("oos_minutes_union")
    naive = frame.groupby("oos_episode_id", sort=False).agg(
        oos_minutes_naive_sum=("_naive_min", "sum"),
        events_with_clear=("_naive_min", "size"),
        events_clear_clamped=("_clamped", "sum"),
    )
    return union.to_frame().join(naive).reset_index()


def ps3v22_source_column_profile(events):
    """Measure the candidate label columns instead of trusting their names.

    The export carries severity, event_type_severity, event_priority and
    requires_service_call. A column whose value is fully determined by
    EVENT_TYPE_ID is a lookup on the event type, not an observation of this
    incident -- modelling it would just be relearning the event type. That is
    measured here and reported, and it gates the promotion below.
    """
    rows = []
    type_col = "event_type_id" if "event_type_id" in events.columns else None
    for column in ["observed_event_severity", "event_type_severity", "event_priority",
                   "requires_service_call", "observed_event_component", "component_subsystem",
                   "event_type_name"]:
        if column not in events.columns:
            rows.append({"column": column, "status": "not_present"})
            continue
        values = events[column]
        non_null = int(values.notna().sum())
        distinct = int(values.nunique(dropna=True))
        deterministic = None
        if type_col is not None and non_null:
            per_type = events.loc[values.notna()].groupby(type_col, dropna=False)[column].nunique()
            deterministic = bool((per_type <= 1).all())
        rows.append({
            "column": column, "status": "present",
            "rows": int(len(values)), "non_null": non_null,
            "null_rate": round(1.0 - non_null / max(len(values), 1), 4),
            "distinct_values": distinct,
            "deterministic_given_event_type": deterministic,
            "usable_as_observed_label": bool(
                non_null > 0 and 2 <= distinct <= 50 and deterministic is False),
            "note": ("A value fully determined by EVENT_TYPE_ID is a lookup on the event type, "
                     "not an observation of the incident."),
        })
    return pd.DataFrame(rows)


def ps3v22_promote_source_severity(facts, profile):
    """Use source-observed severity as a label ONLY when it survives the gate.

    This is an OBSERVATION carried in the source, not an invention, so it does
    not breach the contract. It is stamped observed_source_event_severity so it
    is never confused with a severity linked from PS3 evidence, and it is
    refused outright when the profile shows it is a function of the event type.
    """
    if not PS3V21_CONFIG["USE_SOURCE_SEVERITY"] or profile.empty:
        return facts, {"status": "disabled", "detail": "USE_SOURCE_SEVERITY is false."}
    usable = profile.loc[profile.get("usable_as_observed_label", False) == True]  # noqa: E712
    for candidate in ["observed_event_severity", "event_type_severity"]:
        if candidate in set(usable["column"]) and candidate in facts.columns:
            unlabelled = facts["observed_severity_label"].isna()
            facts.loc[unlabelled, "observed_severity_label"] = facts.loc[unlabelled, candidate]
            facts.loc[unlabelled & facts[candidate].notna(), "severity_status"] = "observed_source_event_severity"
            return facts, {"status": "promoted", "column": candidate,
                           "rows": int(facts["observed_severity_label"].notna().sum()),
                           "detail": ("Severity observed in the source event record. Not linked PS3 "
                                      "evidence, and not a model output.")}
    reasons = profile[["column", "distinct_values", "deterministic_given_event_type"]].to_dict("records") \
        if "distinct_values" in profile.columns else []
    return facts, {"status": "not_promoted",
                   "detail": "No source severity column passed the non-triviality gate.",
                   "candidates": reasons}


def ps3v22_build_arrow_filter(dataset, cols):
    """Push as much of the PS1 predicate into the scan as the file format allows.

    V22 pushed down only is_hardware_oos_event, which left 40,238,324 candidate
    rows and tripped a size cap. Every predicate pushed down here is one that
    never has to be materialised. Each is added independently and the whole
    filter is smoke-tested before use, so an unsupported expression degrades to
    a cheaper filter instead of failing the run.
    """
    import pyarrow.compute as pc
    applied = []
    parts = []

    def _try(name, build):
        try:
            parts.append(build())
            applied.append(name)
        except Exception:
            pass

    def _type(column):
        return dataset.schema.field(column).type

    if cols.get("hardware"):
        _try("is_hardware_oos_event", lambda: (
            pc.field(cols["hardware"]) == True if pa.types.is_boolean(_type(cols["hardware"]))  # noqa: E712
            else pc.equal(pc.utf8_upper(pc.utf8_trim_whitespace(
                pc.cast(pc.field(cols["hardware"]), pa.string()))), "TRUE")))

    if cols.get("state"):
        _try("EVENT_STATE_TYPE_NAME=SET", lambda: pc.equal(
            pc.utf8_upper(pc.utf8_trim_whitespace(pc.field(cols["state"]))), "SET"))

    if cols.get("category"):
        aliases = ["TVM", "TVMS", "GATE", "GATES", "VALIDATOR", "VALIDATORS", "BMV"]
        _try("mars_device_category", lambda: pc.is_in(
            pc.utf8_upper(pc.utf8_trim_whitespace(pc.field(cols["category"]))),
            value_set=pa.array(aliases)))

    if cols.get("event_time"):
        start = pd.Timestamp(PS3V21_CONFIG["DATA_AS_OF_DATE"]) - pd.Timedelta(days=PS3V21_CONFIG["LOOKBACK_DAYS"])
        end = pd.Timestamp(PS3V21_CONFIG["DATA_AS_OF_DATE"]) + pd.Timedelta(days=1)
        _try("EVENT_DTM window", lambda: (
            (pc.field(cols["event_time"]) >= pa.scalar(start.to_pydatetime()))
            & (pc.field(cols["event_time"]) < pa.scalar(end.to_pydatetime()))
        ) if pa.types.is_timestamp(_type(cols["event_time"])) else (_ for _ in ()).throw(TypeError()))

    if not parts:
        return None, []
    combined = parts[0]
    for extra in parts[1:]:
        combined = combined & extra
    # Prove the filter before committing to a full scan. If the engine rejects
    # any expression, fall back predicate by predicate rather than failing.
    while parts:
        try:
            dataset.head(1, filter=combined)
            return combined, applied
        except Exception:
            parts.pop()
            applied.pop()
            if not parts:
                return None, []
            combined = parts[0]
            for extra in parts[1:]:
                combined = combined & extra
    return None, []


def ps3v22_read_arrow_spine(path, source_name, apply_current_device_filter):
    """Return (events_pd, status). Never raises.

    V23: streams record batches instead of materialising the whole scan. Peak
    memory is one batch plus the rows that actually survive the PS1 predicate,
    so the size of the source stops being a failure mode. V22 read the scan in
    one call and guarded it with a row cap, which is how a run died 0.6% over
    an arbitrary limit.
    """
    if not PS3V21_PYARROW_READY:
        return None, {"source": source_name, "status": "unavailable", "reference": path,
                      "detail": "pyarrow is not importable; the JVM-free spine cannot be used."}
    try:
        import pyarrow.dataset as pads
    except Exception as exc:
        return None, {"source": source_name, "status": "unavailable", "reference": path,
                      "detail": "pyarrow.dataset unavailable: " + ps3v21_error_detail(exc)}
    try:
        dataset = pads.dataset(path, format="parquet")
    except Exception as exc:
        return None, {"source": source_name, "status": "unavailable", "reference": path,
                      "detail": ps3v21_error_detail(exc)}

    names = list(dataset.schema.names)
    cols = ps3v22_resolve_columns(names)
    missing = [logical for logical, actual in {
        "device_id": cols["device_id"], "mars_device_category": cols["category"],
        "event timestamp": cols["event_time"], "is_hardware_oos_event": cols["hardware"],
        "EVENT_STATE_TYPE_NAME": cols["state"],
    }.items() if actual is None]
    if missing:
        return None, {"source": source_name, "status": "missing_required_columns", "reference": path,
                      "detail": "Missing PS1 hardware-OOS SET fields: " + ", ".join(missing)}

    projection = list(dict.fromkeys([c for c in cols.values() if c is not None]))
    scan_filter, pushed = ps3v22_build_arrow_filter(dataset, cols)

    kept_frames = []
    scanned = 0
    kept = 0
    try:
        scanner = dataset.scanner(columns=projection, filter=scan_filter,
                                  batch_size=int(PS3V21_CONFIG["PYARROW_BATCH_ROWS"]))
        for batch in scanner.to_batches():
            if batch.num_rows == 0:
                continue
            scanned += batch.num_rows
            piece = ps3v22_normalise_arrow_batch(batch.to_pandas(), cols)
            if len(piece):
                kept_frames.append(piece)
                kept += len(piece)
            if kept > PS3V21_CONFIG["MAX_SPINE_EVENT_ROWS"]:
                return None, {"source": source_name, "status": "too_many_spine_events", "reference": path,
                              "scanned_rows": scanned, "rows": kept,
                              "detail": (f"More than {PS3V21_CONFIG['MAX_SPINE_EVENT_ROWS']:,} events survive the "
                                         "PS1 predicate. Shorten LOOKBACK_DAYS or raise PS3_MAX_SPINE_EVENT_ROWS "
                                         "if the kernel has the memory.")}
            if scanned > PS3V21_CONFIG["PYARROW_MAX_SCAN_ROWS"]:
                return None, {"source": source_name, "status": "scan_runaway_guard", "reference": path,
                              "scanned_rows": scanned, "rows": kept,
                              "detail": f"Scanned more than {PS3V21_CONFIG['PYARROW_MAX_SCAN_ROWS']:,} rows; stopping."}
    except Exception as exc:
        return None, {"source": source_name, "status": "read_failed", "reference": path,
                      "scanned_rows": scanned, "detail": ps3v21_error_detail(exc)}

    if not kept_frames:
        return pd.DataFrame(), {"source": source_name, "status": "loaded", "reference": path,
                                "scanned_rows": scanned, "selected_rows": 0,
                                "pushed_down": ", ".join(pushed) or "none",
                                "detail": "The PS1 predicate matched no rows. Check DATA_AS_OF_DATE and LOOKBACK_DAYS."}

    frame = pd.concat(kept_frames, ignore_index=True)
    del kept_frames
    frame = ps3v22_finalise_arrow_events(frame)
    status = {
        "source": source_name, "status": "loaded", "reference": path,
        "detail": "PS1-compatible: hardware OOS=true AND state=SET; chargeability not read or used. Read without a JVM.",
        "engine": "pyarrow_streaming",
        "pushed_down": ", ".join(pushed) or "none",
        "scanned_rows": scanned,
        "window_start": (pd.Timestamp(PS3V21_CONFIG["DATA_AS_OF_DATE"]) - pd.Timedelta(days=PS3V21_CONFIG["LOOKBACK_DAYS"])).date().isoformat(),
        "window_end": PS3V21_CONFIG["DATA_AS_OF_DATE"],
    }
    if apply_current_device_filter and PS3V21_CONFIG["PS1_CURRENT_DEVICE_FILTER"]:
        before = len(frame)
        frame, filter_status = ps3v22_apply_current_device_filter_arrow(frame)
        status["current_device_filter"] = filter_status
        status["rows_removed_by_current_filter"] = before - len(frame)
    else:
        status["current_device_filter"] = "not_requested"
    status["selected_rows"] = int(len(frame))
    return frame, status


def ps3v22_normalise_arrow_batch(raw, cols):
    """Apply the exact PS1 predicate to one batch and keep the spine columns.

    Same predicate, same order, same semantics as ps3v21_normalise_direct_events.
    V24 carries the columns the 76-column export actually has; V23 kept seven.
    """
    def _s(slot):
        column = cols.get(slot)
        return raw[column].astype("string") if column else pd.Series(pd.NA, index=raw.index, dtype="string")

    frame = pd.DataFrame(index=raw.index)
    frame["device_key"] = _s("device_key")
    frame["device_id"] = raw[cols["device_id"]].astype("string")
    frame["mars_device_category"] = ps3v21_category(raw[cols["category"]])
    frame["event_timestamp"] = ps3v21_safe_datetime(raw[cols["event_time"]])
    frame["oos_event_id"] = _s("event_id")
    frame["is_commanded_oos_event"] = ps3v22_arrow_truth(raw[cols["commanded"]]) if cols.get("commanded") else False
    frame["observed_event_component"] = _s("component")
    frame["component_subsystem"] = _s("subsystem")
    frame["component_position"] = _s("component_pos")
    frame["facility_id"] = _s("facility")
    frame["facility_name"] = _s("facility_name")
    frame["bus_id"] = _s("bus")
    frame["component_serial_nbr"] = _s("serial")
    frame["event_type_id"] = _s("event_type_id")
    frame["event_type_name"] = _s("event_type")
    frame["observed_event_severity"] = _s("severity")
    frame["event_type_severity"] = _s("event_type_sev")
    frame["event_priority"] = _s("priority")
    frame["requires_service_call"] = ps3v22_arrow_truth(raw[cols["service_call"]]) if cols.get("service_call") else False
    frame["automatic_clear_flag"] = ps3v22_arrow_truth(raw[cols["auto_clear"]]) if cols.get("auto_clear") else False
    frame["clear_timestamp"] = ps3v21_safe_datetime(raw[cols["clear_time"]]) if cols.get("clear_time") else pd.NaT
    frame["duration_to_clear_min"] = (pd.to_numeric(raw[cols["duration_clear"]], errors="coerce")
                                      if cols.get("duration_clear") else np.nan)

    hardware_true = ps3v22_arrow_truth(raw[cols["hardware"]])
    state_is_set = raw[cols["state"]].astype("string").str.strip().str.upper().eq("SET").fillna(False)
    start_date = pd.Timestamp(PS3V21_CONFIG["DATA_AS_OF_DATE"]) - pd.Timedelta(days=PS3V21_CONFIG["LOOKBACK_DAYS"])
    end_date = pd.Timestamp(PS3V21_CONFIG["DATA_AS_OF_DATE"])
    event_date = frame["event_timestamp"].dt.normalize()
    keep = (
        hardware_true
        & state_is_set
        & frame["mars_device_category"].isin(["TVM", "GATE", "VALIDATOR"])
        & frame["device_id"].notna()
        & frame["event_timestamp"].notna()
        & event_date.between(start_date.normalize(), end_date.normalize())
    )
    frame = frame.loc[keep]
    if PS3V21_CONFIG["EXCLUDE_COMMANDED_OOS"]:
        frame = frame.loc[~frame["is_commanded_oos_event"]]
    return frame.reset_index(drop=True)


def ps3v22_finalise_arrow_events(frame):
    """Global steps that cannot be done per batch: id backfill and de-duplication."""
    blank = frame["oos_event_id"].isna() | (frame["oos_event_id"].astype("string").str.strip() == "")
    if blank.any():
        synthetic = ps3v22_sha256_series(
            frame.loc[blank].assign(_ts=frame.loc[blank, "event_timestamp"].astype("string")),
            ["device_id", "mars_device_category", "_ts"],
        )
        frame.loc[blank, "oos_event_id"] = synthetic
    frame["oos_event_id"] = frame["oos_event_id"].astype("string").str.strip()
    frame = frame.drop_duplicates(subset=["oos_event_id", "device_id", "mars_device_category", "event_timestamp"])
    return frame.reset_index(drop=True)


def ps3v22_apply_current_device_filter_arrow(frame):
    if not PS3V21_PYARROW_READY:
        return frame, "not_applied_pyarrow_unavailable"
    try:
        import pyarrow.dataset as pads
        dim = pads.dataset(PS3V21_CONFIG["RAW_SILVER_DIM_DEVICE_PATH"], format="parquet")
        names = list(dim.schema.names)
        dim_key = ps3v22_pick_arrow(names, ["device_key", "device_sk"])
        dim_current = ps3v22_pick_arrow(names, ["is_current", "current_flag"])
        if not dim_key or not dim_current:
            return frame, "not_applied_dim_unavailable_or_schema_missing"
        dim_pd = dim.to_table(columns=[dim_key, dim_current]).to_pandas()
        current = dim_pd.loc[ps3v22_arrow_truth(dim_pd[dim_current]), dim_key].astype("string").dropna().unique()
        if frame["device_key"].isna().all():
            return frame, "not_applied_no_device_key_in_events"
        return frame.loc[frame["device_key"].isin(set(current))].copy(), "applied"
    except Exception as exc:
        return frame, "not_applied_" + ps3v21_error_detail(exc)[:120]


def ps3v22_sessionise_events_pandas(events):
    """Byte-identical episode semantics to ps3v21_sessionise_events, in pandas.

    Same partition, same order, same gap rule, same sha256 episode id, so an
    episode keeps its identity whichever engine produced it.

    V24 FIX: the episode aggregation now carries facility, bus, serial, position,
    subsystem, event type and severity. V23 aggregated only six fields, so
    facility_id and component_serial_nbr existed on the event frame and were
    then dropped at episode grain -- which is why ps3_facility_rollup and
    ps3_serial_reliability published one placeholder row on a successful run.
    """
    gap_seconds = int(PS3V21_CONFIG["PS1_OOS_EPISODE_GAP_DAYS"]) * 86400
    frame = events.sort_values(
        ["mars_device_category", "device_id", "event_timestamp", "oos_event_id"], kind="mergesort"
    ).reset_index(drop=True)
    group = [frame["mars_device_category"], frame["device_id"]]
    prior = frame.groupby(group, sort=False, dropna=False)["event_timestamp"].shift(1)
    delta = (frame["event_timestamp"] - prior).dt.total_seconds()
    new_episode = prior.isna() | (delta > gap_seconds)
    frame["_episode_number"] = new_episode.groupby(group, sort=False, dropna=False).cumsum().astype("int64")
    frame["oos_episode_id"] = ps3v22_sha256_series(
        frame.assign(_n=frame["_episode_number"].astype("string")),
        ["mars_device_category", "device_id", "_n"],
    )

    first_non_null = lambda s: s.dropna().iloc[0] if s.notna().any() else pd.NA
    spec = {
        "episode_start": ("event_timestamp", "min"),
        "episode_last_signal": ("event_timestamp", "max"),
        "oos_set_event_count": ("event_timestamp", "size"),
        "first_oos_event_id": ("oos_event_id", "first"),
        "contains_commanded_oos_signal": ("is_commanded_oos_event", "max"),
    }
    for column in PS3V22_EPISODE_FIRST:
        if column in frame.columns:
            spec[column] = (column, first_non_null)
    for column, alias in (("requires_service_call", "requires_service_call"),
                          ("automatic_clear_flag", "any_automatic_clear")):
        if column in frame.columns:
            spec[alias] = (column, "max")
    if "component_serial_nbr" in frame.columns:
        spec["distinct_serials_in_episode"] = ("component_serial_nbr", "nunique")
    if "observed_event_component" in frame.columns:
        spec["distinct_components_in_episode"] = ("observed_event_component", "nunique")

    grouped = frame.groupby(["oos_episode_id", "device_id", "mars_device_category"], sort=False, dropna=False)
    episodes = grouped.agg(**spec).reset_index()
    episodes["oos_set_event_count"] = episodes["oos_set_event_count"].astype("int64")
    episodes["contains_commanded_oos_signal"] = episodes["contains_commanded_oos_signal"].astype(bool)
    episodes["set_signal_span_minutes"] = (
        episodes["episode_last_signal"] - episodes["episode_start"]
    ).dt.total_seconds() / 60.0
    episodes["oos_fact_definition"] = (
        f"PS1 hardware-OOS SET session; a new session starts after > "
        f"{PS3V21_CONFIG['PS1_OOS_EPISODE_GAP_DAYS']} days"
    )
    # Honest downtime per episode: the union of the [event, clear] intervals,
    # never their sum. The naive sum is published beside it so the gap is visible.
    downtime = ps3v22_episode_downtime(frame)
    if not downtime.empty:
        episodes = episodes.merge(downtime, on="oos_episode_id", how="left")
    return episodes


In [5]:
# 4. Optional PS3 labels and Root-Cause Factory — evidence is linked after OOS discovery.

PS3V21_COLUMN_ALIASES = {
    "device_id": ["device_id", "device", "serial_number", "asset_id"],
    "mars_device_category": ["mars_device_category", "device_category", "device_type", "category"],
    "event_timestamp": ["event_timestamp", "event_dtm", "incident_timestamp", "opened_at", "incident_opened_at", "episode_start", "start_dtm"],
    "event_id": ["oos_event_id", "availability_event_id", "device_event_id", "dw_device_event_id", "event_id", "source_event_id"],
    "severity": ["severity", "severity_label", "failure_level", "failure_level_label", "incident_severity", "impact_severity"],
    "component": ["component_type", "component", "device_component", "failure_component", "component_label"],
    "root_cause": ["root_cause", "root_cause_label", "cause", "cause_label", "failure_cause", "cause_description"],
    "root_cause_domain": ["root_cause_domain", "cause_domain", "failure_domain"],
    "approved": ["approved", "is_approved", "taxonomy_approved", "approval_status"],
    "confidence": ["confidence", "evidence_confidence", "root_cause_confidence", "score"],
    "source_value": ["source_value", "raw_label", "source_label", "root_cause", "root_cause_label", "cause", "cause_label"],
}


def ps3v21_pick_pandas(frame, logical):
    lookup = {ps3v21_norm(column): column for column in frame.columns}
    for candidate in PS3V21_COLUMN_ALIASES[logical]:
        value = lookup.get(ps3v21_norm(candidate))
        if value:
            return value
    return None


def ps3v21_normalise_severity(series):
    text = series.astype("string").str.strip().str.upper()
    mapping = {
        "1": "CRITICAL", "P1": "CRITICAL", "SEV1": "CRITICAL", "CRIT": "CRITICAL", "CRITICAL": "CRITICAL",
        "2": "HIGH", "P2": "HIGH", "SEV2": "HIGH", "HIGH": "HIGH",
        "3": "MEDIUM", "P3": "MEDIUM", "SEV3": "MEDIUM", "MED": "MEDIUM", "MEDIUM": "MEDIUM",
        "4": "LOW", "P4": "LOW", "SEV4": "LOW", "LOW": "LOW",
    }
    return text.map(mapping).where(text.notna())


def ps3v21_read_optional_export(spark_session, path, source_name):
    if not ps3v21_text(path):
        return pd.DataFrame(), {"source": source_name, "status": "not_configured", "reference": None, "rows": 0, "detail": "Optional source not configured."}
    if spark_session is None:
        return pd.DataFrame(), {"source": source_name, "status": "not_ready", "reference": path, "rows": 0, "detail": "Spark is unavailable; optional source was not read."}
    raw, read_status = ps3v21_read_spark_parquet(spark_session, path, source_name)
    if raw is None:
        read_status["rows"] = 0
        return pd.DataFrame(), read_status
    try:
        collected = raw.limit(PS3V21_CONFIG["MAX_OPTIONAL_EXPORT_ROWS"] + 1).toPandas()
        if len(collected) > PS3V21_CONFIG["MAX_OPTIONAL_EXPORT_ROWS"]:
            return pd.DataFrame(), {"source": source_name, "status": "too_large_for_local_label_lane", "reference": path, "rows": len(collected), "detail": f"Bound this optional export to <= {PS3V21_CONFIG['MAX_OPTIONAL_EXPORT_ROWS']:,} rows; OOS facts remain valid."}
        return collected, {"source": source_name, "status": "loaded", "reference": path, "rows": len(collected), "detail": None}
    except Exception as exc:
        return pd.DataFrame(), {"source": source_name, "status": "read_failed", "reference": path, "rows": 0, "detail": ps3v21_error_detail(exc)}


def ps3v21_canonical_evidence(raw, source_name, default_confidence):
    if raw.empty:
        return pd.DataFrame(columns=["device_id", "mars_device_category", "event_timestamp", "event_id", "severity", "component", "root_cause", "root_cause_domain", "approved", "confidence", "evidence_source"])
    result = pd.DataFrame(index=raw.index)
    for logical in ["device_id", "mars_device_category", "event_timestamp", "event_id", "severity", "component", "root_cause", "root_cause_domain", "approved", "confidence"]:
        column = ps3v21_pick_pandas(raw, logical)
        result[logical] = raw[column] if column else pd.NA
    result["device_id"] = result["device_id"].astype("string").str.strip()
    result["mars_device_category"] = ps3v21_category(result["mars_device_category"])
    result["event_timestamp"] = ps3v21_safe_datetime(result["event_timestamp"])
    result["event_id"] = result["event_id"].astype("string").str.strip()
    result["severity"] = ps3v21_normalise_severity(result["severity"])
    result["component"] = result["component"].astype("string").str.strip().replace("", pd.NA)
    result["root_cause"] = result["root_cause"].astype("string").str.strip().replace("", pd.NA)
    result["root_cause_domain"] = result["root_cause_domain"].astype("string").str.strip().replace("", pd.NA)
    result["approved"] = ps3v21_safe_bool(result["approved"])
    result["confidence"] = pd.to_numeric(result["confidence"], errors="coerce").fillna(float(default_confidence)).clip(0.0, 1.0)
    result["evidence_source"] = source_name
    return result.loc[result["device_id"].notna() & result["event_timestamp"].notna()].reset_index(drop=True)


def ps3v21_taxonomy_map(raw):
    if raw.empty:
        return pd.DataFrame(columns=["root_cause_key", "taxonomy_root_cause", "taxonomy_domain", "taxonomy_component", "taxonomy_approved"])
    source = ps3v21_pick_pandas(raw, "source_value")
    root = ps3v21_pick_pandas(raw, "root_cause")
    domain = ps3v21_pick_pandas(raw, "root_cause_domain")
    component = ps3v21_pick_pandas(raw, "component")
    approved = ps3v21_pick_pandas(raw, "approved")
    if source is None and root is None:
        return pd.DataFrame(columns=["root_cause_key", "taxonomy_root_cause", "taxonomy_domain", "taxonomy_component", "taxonomy_approved"])
    key_values = raw[source] if source else raw[root]
    result = pd.DataFrame({
        "root_cause_key": key_values.astype("string").map(ps3v21_norm),
        "taxonomy_root_cause": raw[root].astype("string").str.strip() if root else key_values.astype("string").str.strip(),
        "taxonomy_domain": raw[domain].astype("string").str.strip() if domain else pd.NA,
        "taxonomy_component": raw[component].astype("string").str.strip() if component else pd.NA,
        "taxonomy_approved": ps3v21_safe_bool(raw[approved]) if approved else False,
    })
    return result.loc[result["root_cause_key"].ne("")].drop_duplicates("root_cause_key", keep="last")


def ps3v21_link_evidence(episodes, evidence, tolerance_hours):
    """Conservative exact-ID then same-device nearest-time evidence linkage."""
    output_columns = ["_episode_row_id", "linked_severity", "linked_component", "linked_root_cause", "linked_root_cause_domain", "linked_confidence", "linked_source", "link_method"]
    if episodes.empty or evidence.empty:
        return pd.DataFrame(columns=output_columns)
    left = episodes[["_episode_row_id", "device_id", "mars_device_category", "episode_start", "first_oos_event_id"]].copy()
    left["device_id"] = left["device_id"].astype("string")
    right = evidence.copy()
    right["device_id"] = right["device_id"].astype("string")
    right = right.sort_values(["confidence", "event_timestamp"], ascending=[False, False]).drop_duplicates(["event_id"], keep="first")
    rows = []
    # Exact OOS event IDs are strongest. Evidence IDs that are missing are not
    # allowed to match each other.
    exact_right = right.loc[right["event_id"].notna() & right["event_id"].ne("")].copy()
    if not exact_right.empty:
        exact = left.merge(exact_right, left_on="first_oos_event_id", right_on="event_id", how="left", suffixes=("", "_ev"))
        exact = exact.loc[exact["evidence_source"].notna()].copy()
        if not exact.empty:
            exact["link_method"] = "exact_oos_event_id"
            rows.append(exact)
    matched_ids = set(pd.concat(rows)["_episode_row_id"].tolist()) if rows else set()
    remaining = left.loc[~left["_episode_row_id"].isin(matched_ids)].copy()
    if not remaining.empty:
        # `merge_asof` uses only historical/near-time evidence for the same
        # device. Category mismatch is rejected after matching (rather than
        # silently cross-linking TVM/GATE/VALIDATOR rows).
        try:
            lsort = remaining.sort_values("episode_start")
            rsort = right.sort_values("event_timestamp").rename(columns={"mars_device_category": "evidence_category"})
            nearest = pd.merge_asof(
                lsort, rsort, left_on="episode_start", right_on="event_timestamp", by="device_id",
                direction="nearest", tolerance=pd.Timedelta(hours=int(tolerance_hours)), suffixes=("", "_ev"),
            )
            cat_ok = nearest["evidence_category"].isna() | nearest["evidence_category"].eq("") | nearest["evidence_category"].eq(nearest["mars_device_category"])
            nearest = nearest.loc[nearest["evidence_source"].notna() & cat_ok].copy()
            if not nearest.empty:
                nearest["link_method"] = "same_device_nearest_time"
                rows.append(nearest)
        except Exception:
            # A problematic optional evidence export must never fail the OOS
            # fact lane. The source audit records it as unmatched instead.
            pass
    if not rows:
        return pd.DataFrame(columns=output_columns)
    linked = pd.concat(rows, ignore_index=True, sort=False)
    linked = linked.sort_values(["_episode_row_id", "confidence"], ascending=[True, False]).drop_duplicates("_episode_row_id", keep="first")
    linked = linked.rename(columns={
        "severity": "linked_severity", "component": "linked_component", "root_cause": "linked_root_cause",
        "root_cause_domain": "linked_root_cause_domain", "confidence": "linked_confidence", "evidence_source": "linked_source",
    })
    return linked[[column for column in output_columns if column in linked.columns]]


def ps3v21_build_root_cause_factory(episodes, spark_session):
    configs = [
        ("gold_incident_labels", PS3V21_CONFIG["GOLD_INCIDENT_LABEL_EXPORT"], 0.70),
        ("incident_root_cause", PS3V21_CONFIG["INCIDENT_ROOT_CAUSE_EXPORT"], 0.90),
        ("maintenance_ledger", PS3V21_CONFIG["MAINTENANCE_LEDGER_EXPORT"], 0.80),
        ("incident_task_ci", PS3V21_CONFIG["INCIDENT_TASK_CI_EXPORT"], 0.75),
    ]
    audit = []
    evidence_parts = []
    for name, path, default_confidence in configs:
        raw, status = ps3v21_read_optional_export(spark_session, path, name)
        audit.append(status)
        if not raw.empty:
            evidence_parts.append(ps3v21_canonical_evidence(raw, name, default_confidence))
    tax_raw, tax_status = ps3v21_read_optional_export(spark_session, PS3V21_CONFIG["ROOTCAUSE_TAXONOMY_EXPORT"], "root_cause_taxonomy")
    audit.append(tax_status)
    taxonomy = ps3v21_taxonomy_map(tax_raw)
    evidence = pd.concat(evidence_parts, ignore_index=True, sort=False) if evidence_parts else pd.DataFrame(columns=["device_id", "mars_device_category", "event_timestamp", "event_id", "severity", "component", "root_cause", "root_cause_domain", "approved", "confidence", "evidence_source"])
    if not evidence.empty:
        evidence["root_cause_key"] = evidence["root_cause"].astype("string").map(ps3v21_norm)
        if not taxonomy.empty:
            evidence = evidence.merge(taxonomy, on="root_cause_key", how="left")
            evidence["confirmed_root_cause"] = evidence["taxonomy_root_cause"].where(evidence["taxonomy_approved"], pd.NA)
            evidence["confirmed_domain"] = evidence["taxonomy_domain"].where(evidence["taxonomy_approved"], pd.NA)
            evidence["confirmed_component"] = evidence["taxonomy_component"].where(evidence["taxonomy_approved"], pd.NA)
        else:
            evidence["taxonomy_approved"] = False
            evidence["confirmed_root_cause"] = pd.NA
            evidence["confirmed_domain"] = pd.NA
            evidence["confirmed_component"] = pd.NA
    return evidence, taxonomy, audit


def ps3v21_enrich_episodes(episodes, spark_session):
    facts = episodes.copy().reset_index(drop=True)
    facts["_episode_row_id"] = np.arange(len(facts), dtype="int64")
    facts["device_id"] = facts["device_id"].astype("string")
    facts["mars_device_category"] = ps3v21_category(facts["mars_device_category"])
    facts["episode_start"] = ps3v21_safe_datetime(facts["episode_start"])
    facts["episode_last_signal"] = ps3v21_safe_datetime(facts["episode_last_signal"])
    facts["observed_event_component"] = facts.get("observed_event_component", pd.Series(pd.NA, index=facts.index)).astype("string").replace("", pd.NA)
    evidence, taxonomy, source_audit = ps3v21_build_root_cause_factory(facts, spark_session)
    linked = ps3v21_link_evidence(facts, evidence, PS3V21_CONFIG["EVIDENCE_LINK_TOLERANCE_HOURS"])
    facts = facts.merge(linked, on="_episode_row_id", how="left")
    facts["observed_severity_label"] = facts.get("linked_severity", pd.Series(pd.NA, index=facts.index))
    facts["severity_status"] = np.where(facts["observed_severity_label"].notna(), "observed_linked_label", "unavailable_no_linked_label")
    # V22: smoke-test ONLY. Without a label the model, SHAP and scorecard
    # lanes can never run, so a smoke test proves nothing beyond "the
    # notebook started". Hard-gated on SMOKE_TEST_SYNTHETIC, which also
    # forces PUBLISH_TO_S3 off, so a synthetic label cannot reach a table.
    if PS3V21_CONFIG["SMOKE_TEST_SYNTHETIC"] and "_smoke_severity" in facts.columns:
        facts["observed_severity_label"] = facts["_smoke_severity"]
        facts["severity_status"] = "synthetic_smoke_test_label_not_for_delivery"

    facts["component_attribution"] = facts.get("confirmed_component", pd.Series(pd.NA, index=facts.index)) if "confirmed_component" in facts else pd.NA
    # The top linked source may carry a raw component; this is component
    # attribution, not a confirmed root cause unless taxonomy evidence exists.
    if "linked_component" in facts:
        facts["component_attribution"] = facts["component_attribution"].fillna(facts["linked_component"])
    facts["component_attribution"] = facts["component_attribution"].fillna(facts["observed_event_component"])
    facts["component_attribution_status"] = np.where(
        facts.get("linked_source", pd.Series(pd.NA, index=facts.index)).notna(), "linked_component_attribution",
        np.where(facts["observed_event_component"].notna(), "event_component_observation", "unavailable"),
    )
    # Root-cause confirmation requires approved taxonomy mapping + evidence.
    facts["confirmed_root_cause_label"] = pd.NA
    facts["confirmed_root_cause_domain"] = pd.NA
    facts["root_cause_confidence"] = pd.NA
    facts["root_cause_evidence_source"] = facts.get("linked_source", pd.Series(pd.NA, index=facts.index))
    facts["root_cause_link_method"] = facts.get("link_method", pd.Series(pd.NA, index=facts.index))
    if not evidence.empty and not taxonomy.empty:
        approved_evidence = evidence.loc[evidence["taxonomy_approved"] & evidence["confirmed_root_cause"].notna()].copy()
        approved_evidence["root_cause"] = approved_evidence["confirmed_root_cause"]
        approved_evidence["root_cause_domain"] = approved_evidence["confirmed_domain"]
        approved_evidence["component"] = approved_evidence["confirmed_component"]
        approved_links = ps3v21_link_evidence(facts, approved_evidence, PS3V21_CONFIG["EVIDENCE_LINK_TOLERANCE_HOURS"])
        if not approved_links.empty:
            # Conservative conflict policy: only accept an approved mapped
            # cause if the linked evidence resolves to one high-confidence
            # candidate. Conflicts remain visible but unconfirmed.
            approved_links = approved_links.rename(columns={
                "linked_root_cause": "approved_root_cause_label",
                "linked_root_cause_domain": "approved_root_cause_domain",
                "linked_component": "approved_component",
                "linked_confidence": "approved_root_cause_confidence",
                "linked_source": "approved_root_cause_evidence_source",
                "link_method": "approved_root_cause_link_method",
            })
            facts = facts.merge(approved_links, on="_episode_row_id", how="left")
            facts["confirmed_root_cause_label"] = facts.get("approved_root_cause_label", pd.Series(pd.NA, index=facts.index))
            facts["confirmed_root_cause_domain"] = facts.get("approved_root_cause_domain", pd.Series(pd.NA, index=facts.index))
            facts["root_cause_confidence"] = facts.get("approved_root_cause_confidence", pd.Series(pd.NA, index=facts.index))
            facts["root_cause_evidence_source"] = facts.get("approved_root_cause_evidence_source", facts["root_cause_evidence_source"])
            facts["root_cause_link_method"] = facts.get("approved_root_cause_link_method", facts["root_cause_link_method"])
    has_confirmed = facts["confirmed_root_cause_label"].notna() & facts["confirmed_root_cause_domain"].notna()
    has_candidate = facts.get("linked_root_cause", pd.Series(pd.NA, index=facts.index)).notna()
    facts["root_cause_status"] = np.where(has_confirmed, "confirmed_evidence_and_taxonomy", np.where(has_candidate, "candidate_evidence_taxonomy_not_confirmed", "unconfirmed_no_evidence"))
    facts["candidate_root_cause_raw"] = facts.get("linked_root_cause", pd.Series(pd.NA, index=facts.index))
    facts["evidence_conflict_status"] = "single_best_link_or_conflict_not_evaluable_from_bounded_evidence_export"
    return facts, source_audit, taxonomy


In [6]:
# 5. Leakage-safe features, model quality gates, and SHAP model drivers.

def ps3v21_add_pre_event_features(facts):
    """Leakage-safe pre-event features.

    V22: same outputs as V21, computed on a single sorted pass with numpy
    searchsorted per device slice and ONE assignment back into the frame.
    V21 wrote `frame.loc[group.index, col] = ...` into the full frame once per
    device, which is what made this the wall-clock bottleneck.
    """
    frame = facts.copy()
    frame["event_month"] = frame["episode_start"].dt.month.astype("Int64")
    frame["event_day_of_week"] = frame["episode_start"].dt.dayofweek.astype("Int64")
    frame["event_hour"] = frame["episode_start"].dt.hour.astype("Int64")
    frame["log_oos_set_event_count"] = np.log1p(pd.to_numeric(frame["oos_set_event_count"], errors="coerce").fillna(0).clip(lower=0))
    frame["log_set_signal_span_minutes"] = np.log1p(pd.to_numeric(frame.get("set_signal_span_minutes", 0), errors="coerce").fillna(0).clip(lower=0))

    n = len(frame)
    prior = {days: np.zeros(n, dtype="float64") for days in (7, 30, 90)}
    since = np.full(n, np.nan, dtype="float64")
    recency = np.array(["first_observed_episode"] * n, dtype=object)

    order = frame.sort_values(["mars_device_category", "device_id", "episode_start"], kind="mergesort").index
    positions = frame.index.get_indexer(order)
    ordered = frame.loc[order]
    keys = list(zip(ordered["mars_device_category"].astype("string").fillna(""), ordered["device_id"].astype("string").fillna("")))
    stamps = ordered["episode_start"].astype("int64", copy=False).to_numpy()

    start = 0
    for index in range(1, n + 1):
        if index < n and keys[index] == keys[start]:
            continue
        slice_stamps = stamps[start:index]
        slice_positions = positions[start:index]
        left = np.searchsorted(slice_stamps, slice_stamps, side="left")
        for days in (7, 30, 90):
            window_start = np.searchsorted(slice_stamps, slice_stamps - days * 86400 * 10 ** 9, side="left")
            prior[days][slice_positions] = np.maximum(0, left - window_start)
        if len(slice_stamps) > 1:
            gaps = np.diff(slice_stamps) / (86400 * 10 ** 9)
            since[slice_positions[1:]] = gaps
            recency[slice_positions[1:]] = "prior_episode_available"
        start = index

    for days in (7, 30, 90):
        frame[f"prior_episodes_{days}d"] = prior[days]
    frame["days_since_prior_episode"] = since
    frame["device_oos_recency_status"] = recency
    return frame


def ps3v21_add_feature_snapshot(facts, spark_session, path, source_name):
    audit = {"source": source_name, "status": "not_configured", "detail": "Optional pre-event feature snapshot not configured."}
    if not ps3v21_text(path):
        return facts, audit
    raw, status = ps3v21_read_optional_export(spark_session, path, source_name)
    audit = status
    if raw.empty:
        return facts, audit
    device_col = ps3v21_pick_pandas(raw, "device_id")
    timestamp_col = ps3v21_pick_pandas(raw, "event_timestamp")
    if device_col is None or timestamp_col is None:
        audit.update({"status": "missing_required_columns", "detail": "Feature export needs device_id and a point-in-time timestamp."})
        return facts, audit
    feature = pd.DataFrame({"device_id": raw[device_col].astype("string"), "feature_timestamp": ps3v21_safe_datetime(raw[timestamp_col])})
    forbidden = ("target", "label", "rootcause", "severity", "charge", "will", "future", "outcome", "prediction", "probability")
    numeric = []
    for column in raw.columns:
        key = ps3v21_norm(column)
        if column in {device_col, timestamp_col} or any(token in key for token in forbidden):
            continue
        converted = pd.to_numeric(raw[column], errors="coerce")
        if converted.notna().mean() >= 0.10:
            name = f"{source_name}_{ps3v21_norm(column)[:60]}"
            feature[name] = converted
            numeric.append(name)
        if len(numeric) >= 40:
            break
    if not numeric:
        audit.update({"status": "no_safe_numeric_features", "detail": "No non-leaky numeric columns were available."})
        return facts, audit
    feature = feature.dropna(subset=["device_id", "feature_timestamp"]).sort_values("feature_timestamp")
    left = facts.sort_values("episode_start").copy()
    try:
        joined = pd.merge_asof(
            left, feature, left_on="episode_start", right_on="feature_timestamp", by="device_id",
            direction="backward", tolerance=pd.Timedelta(days=7),
        ).sort_values("_episode_row_id")
        audit.update({"status": "loaded", "safe_feature_count": len(numeric), "detail": "Joined point-in-time feature snapshots only from <= episode start."})
        return joined, audit
    except Exception as exc:
        audit.update({"status": "join_failed", "detail": ps3v21_error_detail(exc)})
        return facts, audit


def ps3v21_make_preprocessor(numeric_columns, categorical_columns):
    try:
        onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        onehot = OneHotEncoder(handle_unknown="ignore", sparse=False)
    return ColumnTransformer([
        ("numeric", Pipeline([("impute", SimpleImputer(strategy="median"))]), numeric_columns),
        ("categorical", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", onehot)]), categorical_columns),
    ], remainder="drop")


def ps3v21_train_target_model(facts, target_column, model_key):
    empty_scorecard = pd.DataFrame(columns=["target", "candidate_model", "f1_macro", "f1_weighted", "balanced_accuracy", "accuracy", "mcc", "majority_f1_macro", "macro_f1_lift", "quality_gate", "detail"])
    if not PS3V21_SKLEARN_READY:
        return {"status": "skipped_dependency", "detail": globals().get("PS3V21_SKLEARN_ERROR"), "scorecard": empty_scorecard, "feature_importance": pd.DataFrame(), "explainability": pd.DataFrame(), "scored": facts}
    labelled = facts.loc[facts[target_column].notna() & facts["episode_start"].notna()].copy()
    labelled[target_column] = labelled[target_column].astype("string")
    if len(labelled) < PS3V21_CONFIG["MIN_TRAIN_ROWS"]:
        return {"status": "skipped_insufficient_labels", "detail": f"{len(labelled)} labelled rows; need {PS3V21_CONFIG['MIN_TRAIN_ROWS']}.", "scorecard": empty_scorecard, "feature_importance": pd.DataFrame(), "explainability": pd.DataFrame(), "scored": facts}
    cutoff = labelled["episode_start"].quantile(0.75)
    train = labelled.loc[labelled["episode_start"] < cutoff].copy()
    valid = labelled.loc[labelled["episode_start"] >= cutoff].copy()
    if train.empty or valid.empty:
        return {"status": "skipped_no_temporal_split", "detail": "Cannot make a non-empty temporal validation split.", "scorecard": empty_scorecard, "feature_importance": pd.DataFrame(), "explainability": pd.DataFrame(), "scored": facts}
    train_support = train[target_column].value_counts()
    valid_support = valid[target_column].value_counts()
    supported = sorted(set(train_support[train_support >= PS3V21_CONFIG["MIN_TRAIN_CLASS_ROWS"]].index).intersection(valid_support[valid_support >= PS3V21_CONFIG["MIN_VALID_CLASS_ROWS"]].index))
    coverage = float(labelled[target_column].isin(supported).mean()) if len(labelled) else 0.0
    if len(supported) < 2:
        return {"status": "skipped_insufficient_temporal_class_support", "detail": f"No two classes meet train>={PS3V21_CONFIG['MIN_TRAIN_CLASS_ROWS']} and validation>={PS3V21_CONFIG['MIN_VALID_CLASS_ROWS']} support. Label coverage={coverage:.1%}.", "scorecard": empty_scorecard, "feature_importance": pd.DataFrame(), "explainability": pd.DataFrame(), "scored": facts}
    train = train.loc[train[target_column].isin(supported)].copy()
    valid = valid.loc[valid[target_column].isin(supported)].copy()
    candidate_numeric = [
        "event_month", "event_day_of_week", "event_hour", "log_oos_set_event_count", "log_set_signal_span_minutes",
        "prior_episodes_7d", "prior_episodes_30d", "prior_episodes_90d", "days_since_prior_episode",
    ]
    candidate_numeric += [c for c in facts.columns if c.startswith(("ps1_feature_", "ps2_feature_"))]
    numeric_columns = [c for c in candidate_numeric if c in facts.columns and pd.to_numeric(facts[c], errors="coerce").notna().any()]
    categorical_columns = [c for c in ["mars_device_category", "device_oos_recency_status"] if c in facts.columns and facts[c].notna().any()]
    feature_columns = numeric_columns + categorical_columns
    if not feature_columns:
        return {"status": "skipped_no_safe_features", "detail": "No pre-event features are available.", "scorecard": empty_scorecard, "feature_importance": pd.DataFrame(), "explainability": pd.DataFrame(), "scored": facts}
    X_train, y_train = train[feature_columns], train[target_column]
    X_valid, y_valid = valid[feature_columns], valid[target_column]
    candidates = {
        "ExtraTrees_balanced": ExtraTreesClassifier(n_estimators=300, min_samples_leaf=2, max_depth=18, class_weight="balanced", random_state=42, n_jobs=-1),
        "RandomForest_balanced": RandomForestClassifier(n_estimators=350, min_samples_leaf=2, max_depth=18, class_weight="balanced_subsample", random_state=42, n_jobs=-1),
    }
    score_rows, fitted = [], {}
    majority = y_train.mode().iloc[0]
    majority_pred = np.repeat(majority, len(y_valid))
    majority_macro = f1_score(y_valid, majority_pred, labels=supported, average="macro", zero_division=0)
    for candidate_name, estimator in candidates.items():
        try:
            pipe = Pipeline([("preprocess", ps3v21_make_preprocessor(numeric_columns, categorical_columns)), ("model", estimator)])
            pipe.fit(X_train, y_train)
            prediction = pipe.predict(X_valid)
            macro = f1_score(y_valid, prediction, labels=supported, average="macro", zero_division=0)
            weighted = f1_score(y_valid, prediction, labels=supported, average="weighted", zero_division=0)
            balanced = balanced_accuracy_score(y_valid, prediction)
            accuracy = accuracy_score(y_valid, prediction)
            mcc = matthews_corrcoef(y_valid, prediction) if len(set(prediction)) > 1 else 0.0
            passed = macro >= PS3V21_CONFIG["MIN_MACRO_F1"] and (macro - majority_macro) >= PS3V21_CONFIG["MIN_MACRO_F1_LIFT"] and balanced >= PS3V21_CONFIG["MIN_BALANCED_ACCURACY"]
            score_rows.append({"target": target_column, "candidate_model": candidate_name, "f1_macro": macro, "f1_weighted": weighted, "balanced_accuracy": balanced, "accuracy": accuracy, "mcc": mcc, "majority_f1_macro": majority_macro, "macro_f1_lift": macro - majority_macro, "label_coverage": coverage, "quality_gate": "passed" if passed else "not_passed", "detail": f"Temporal split at {cutoff}; supported classes={len(supported)}"})
            fitted[candidate_name] = pipe
        except Exception as exc:
            score_rows.append({"target": target_column, "candidate_model": candidate_name, "f1_macro": np.nan, "f1_weighted": np.nan, "balanced_accuracy": np.nan, "accuracy": np.nan, "mcc": np.nan, "majority_f1_macro": majority_macro, "macro_f1_lift": np.nan, "label_coverage": coverage, "quality_gate": "error", "detail": ps3v21_error_detail(exc)})
    scorecard = pd.DataFrame(score_rows)
    eligible = scorecard.loc[scorecard["quality_gate"].eq("passed")].sort_values("f1_macro", ascending=False)
    if eligible.empty:
        return {"status": "not_publishable_quality_gate", "detail": "No model exceeded the majority baseline and temporal macro-F1 gates.", "scorecard": scorecard, "feature_importance": pd.DataFrame(), "explainability": pd.DataFrame(), "scored": facts}
    chosen_name = eligible.iloc[0]["candidate_model"]
    chosen = fitted[chosen_name]
    scored = facts.copy()
    probabilities = chosen.predict_proba(scored[feature_columns])
    classes = chosen.named_steps["model"].classes_
    top_index = probabilities.argmax(axis=1)
    scored[f"predicted_{model_key}"] = pd.Series(classes[top_index], index=scored.index, dtype="string")
    scored[f"predicted_{model_key}_confidence"] = probabilities.max(axis=1)
    scored[f"{model_key}_model_status"] = "quality_gated_temporal_model"
    preprocessor = chosen.named_steps["preprocess"]
    classifier = chosen.named_steps["model"]
    try:
        names = preprocessor.get_feature_names_out()
        importance = pd.DataFrame({"target": target_column, "model": chosen_name, "feature": names, "importance": classifier.feature_importances_}).sort_values("importance", ascending=False)
    except Exception:
        importance = pd.DataFrame()
    explanations = ps3v21_compute_shap(chosen, scored, feature_columns, model_key, target_column)
    return {"status": "publishable", "detail": f"{chosen_name} passed temporal macro-F1 and majority-baseline gates.", "scorecard": scorecard, "feature_importance": importance, "explainability": explanations, "scored": scored, "chosen_model": chosen_name}


def ps3v21_compute_shap(pipeline, scored, feature_columns, model_key, target_column):
    columns = ["target", "model_output", "oos_episode_id", "device_id", "mars_device_category", "predicted_label", "feature", "shap_value", "abs_shap_value", "explanation_status", "explanation_note"]
    sample = scored.head(PS3V21_CONFIG["MAX_SHAP_EPISODES"]).copy()
    if sample.empty:
        return pd.DataFrame(columns=columns)
    try:
        import shap
    except Exception:
        return pd.DataFrame([{ "target": target_column, "model_output": model_key, "oos_episode_id": pd.NA, "device_id": pd.NA, "mars_device_category": pd.NA, "predicted_label": pd.NA, "feature": pd.NA, "shap_value": np.nan, "abs_shap_value": np.nan, "explanation_status": "not_computed_dependency_unavailable", "explanation_note": "Install/import shap to calculate model-driver SHAP values. SHAP is not physical causation." }])
    try:
        pre = pipeline.named_steps["preprocess"]
        model = pipeline.named_steps["model"]
        transformed = np.asarray(pre.transform(sample[feature_columns]))
        feature_names = list(pre.get_feature_names_out())
        predicted = sample[f"predicted_{model_key}"].astype(str).tolist()
        class_to_idx = {str(c): i for i, c in enumerate(model.classes_)}
        values = shap.TreeExplainer(model).shap_values(transformed)
        rows = []
        for row_pos in range(len(sample)):
            class_idx = class_to_idx.get(predicted[row_pos], 0)
            if isinstance(values, list):
                vector = np.asarray(values[class_idx])[row_pos]
            else:
                array = np.asarray(values)
                if array.ndim == 3 and array.shape[2] == len(model.classes_):
                    vector = array[row_pos, :, class_idx]
                elif array.ndim == 3 and array.shape[1] == len(model.classes_):
                    vector = array[row_pos, class_idx, :]
                else:
                    vector = array[row_pos]
            top = np.argsort(np.abs(vector))[-5:][::-1]
            for idx in top:
                rows.append({"target": target_column, "model_output": model_key, "oos_episode_id": sample.iloc[row_pos]["oos_episode_id"], "device_id": sample.iloc[row_pos]["device_id"], "mars_device_category": sample.iloc[row_pos]["mars_device_category"], "predicted_label": predicted[row_pos], "feature": feature_names[idx], "shap_value": float(vector[idx]), "abs_shap_value": float(abs(vector[idx])), "explanation_status": "computed", "explanation_note": "SHAP describes this model output; it is not a confirmed physical root cause."})
        return pd.DataFrame(rows, columns=columns)
    except Exception as exc:
        return pd.DataFrame([{ "target": target_column, "model_output": model_key, "oos_episode_id": pd.NA, "device_id": pd.NA, "mars_device_category": pd.NA, "predicted_label": pd.NA, "feature": pd.NA, "shap_value": np.nan, "abs_shap_value": np.nan, "explanation_status": "not_computed_error", "explanation_note": ps3v21_error_detail(exc) + "; SHAP is not physical causation." }])


# ---------------------------------------------------------------------------
# V22 ADDITION: per-category training. A pooled gate can pass on the strength
# of one fleet and ship a model that is useless for another, so every target is
# trained per device category first; the pooled model is kept only as a labelled
# fallback when a category cannot meet its own support floor.
# ---------------------------------------------------------------------------
def ps3v22_train_target_by_scope(facts, target_column, model_key):
    runs = []
    scored_columns = {}
    scorecards, importances, explanations = [], [], []

    def _record(result, scope):
        runs.append({"target": target_column, "model_key": model_key, "scope": scope,
                     "status": result["status"], "detail": result["detail"]})
        for frame, sink in ((result["scorecard"], scorecards),
                            (result["feature_importance"], importances),
                            (result["explainability"], explanations)):
            if isinstance(frame, pd.DataFrame) and not frame.empty:
                sink.append(frame.assign(model_scope=scope))

    if PS3V21_CONFIG["MODEL_PER_CATEGORY"]:
        for category in sorted(facts["mars_device_category"].dropna().unique().tolist()):
            subset = facts.loc[facts["mars_device_category"] == category]
            if len(subset) < PS3V21_CONFIG["MIN_CATEGORY_TRAIN_ROWS"]:
                runs.append({"target": target_column, "model_key": model_key, "scope": category,
                             "status": "skipped_insufficient_category_rows",
                             "detail": f"{len(subset)} episodes; need {PS3V21_CONFIG['MIN_CATEGORY_TRAIN_ROWS']}."})
                continue
            result = ps3v21_train_target_model(subset, target_column, model_key)
            _record(result, category)
            if result["status"] == "publishable":
                for column in result["scored"].columns:
                    if column.startswith(f"predicted_{model_key}") or column == f"{model_key}_model_status":
                        scored_columns.setdefault(column, pd.Series(pd.NA, index=facts.index, dtype="object"))
                        scored_columns[column].loc[result["scored"].index] = result["scored"][column]

    covered = {r["scope"] for r in runs if r["status"] == "publishable"}
    uncovered = sorted(set(facts["mars_device_category"].dropna().unique().tolist()) - covered)
    if uncovered or not PS3V21_CONFIG["MODEL_PER_CATEGORY"]:
        pooled_input = facts if not PS3V21_CONFIG["MODEL_PER_CATEGORY"] else facts.loc[facts["mars_device_category"].isin(uncovered)]
        pooled = ps3v21_train_target_model(pooled_input, target_column, model_key)
        _record(pooled, "POOLED_FALLBACK" if PS3V21_CONFIG["MODEL_PER_CATEGORY"] else "POOLED")
        if pooled["status"] == "publishable":
            for column in pooled["scored"].columns:
                if column.startswith(f"predicted_{model_key}") or column == f"{model_key}_model_status":
                    scored_columns.setdefault(column, pd.Series(pd.NA, index=facts.index, dtype="object"))
                    scored_columns[column].loc[pooled["scored"].index] = pooled["scored"][column]

    return {
        "runs": runs,
        "scored_columns": scored_columns,
        "scorecard": pd.concat(scorecards, ignore_index=True, sort=False) if scorecards else pd.DataFrame(),
        "feature_importance": pd.concat(importances, ignore_index=True, sort=False) if importances else pd.DataFrame(),
        "explainability": pd.concat(explanations, ignore_index=True, sort=False) if explanations else pd.DataFrame(),
        "any_publishable": bool(covered) or any(r["status"] == "publishable" for r in runs),
    }


# ---------------------------------------------------------------------------
# V22 ADDITION: bounded observational causal layer.
#
# WHAT IT ASKS. For each component attribution with enough support: after an
# episode attributed to that component, is a repeat OOS episode on the same
# device within H days more or less likely than for comparable episodes?
#
# WHY THIS SHAPE. Treatment and outcome both come from the OOS spine PS3
# already has, so the layer adds no new source and cannot contradict PS1/PS2.
# The estimator is AIPW (doubly robust) cross-fitted over TIME folds, never
# random folds, so a later period is never used to explain an earlier one.
#
# WHAT IT IS NOT. Observational. Confounding by unmeasured maintenance policy
# is not ruled out and is stated on every row. It is not an experiment and it
# does not license a causal claim on its own.
# ---------------------------------------------------------------------------
def ps3v22_causal_confounders(facts):
    numeric = [c for c in ["prior_episodes_7d", "prior_episodes_30d", "prior_episodes_90d",
                           "days_since_prior_episode", "log_oos_set_event_count",
                           "log_set_signal_span_minutes", "event_month", "event_day_of_week", "event_hour"]
               if c in facts.columns]
    categorical = [c for c in ["mars_device_category"] if c in facts.columns]
    return numeric, categorical


def ps3v22_repeat_outcome(facts, horizon_days):
    """1 if the same device has another episode starting within H days."""
    frame = facts[["device_id", "mars_device_category", "episode_start"]].copy()
    frame["_row"] = np.arange(len(frame))
    ordered = frame.sort_values(["mars_device_category", "device_id", "episode_start"], kind="mergesort")
    next_start = ordered.groupby(["mars_device_category", "device_id"], sort=False, dropna=False)["episode_start"].shift(-1)
    gap_days = (next_start - ordered["episode_start"]).dt.total_seconds() / 86400.0
    outcome = pd.Series(np.nan, index=frame.index, dtype="float64")
    outcome.iloc[ordered["_row"].to_numpy()] = np.where(gap_days.notna(), (gap_days <= horizon_days).astype(float), 0.0)
    days_to_next = pd.Series(np.nan, index=frame.index, dtype="float64")
    days_to_next.iloc[ordered["_row"].to_numpy()] = gap_days.to_numpy()
    return outcome, days_to_next


def ps3v22_aipw_effect(frame, treated_mask, outcome, numeric, categorical, folds):
    """Cross-fitted AIPW ATE with overlap and balance diagnostics.

    V25 fixes three things the V24 run exposed:

    1. CALIBRATION, and this is the important one. V24 fitted the propensity
       model with class_weight="balanced". Balancing deliberately distorts the
       predicted probabilities toward the minority class -- which is fine for a
       classifier judged on recall, and wrong for a propensity score, because
       AIPW divides by that probability. A rare component's weights were
       therefore systematically wrong. The propensity model is now unweighted,
       so its output is an estimate of P(treated | X) rather than a
       recall-tuned score.
    2. CONVERGENCE. lbfgs hit its 400-iteration limit repeatedly on unscaled
       features. Numeric features are now standardised and max_iter is 2000.
    3. OVERLAP MEASURED AGAINST PREVALENCE. A fixed [0.02, 0.98] band declares
       almost no overlap for a treatment with 1% prevalence, purely because true
       propensities sit near 0.01 -- which is what produced overlap shares of
       0.089 and 0.104 on components with ~600-1200 treated episodes. Overlap is
       now the share of units inside the common-support region actually shared
       by the treated and control propensity distributions, which adapts to
       prevalence instead of assuming it.
    """
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    design = frame[numeric + categorical]
    pre = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), numeric),
        ("cat", Pipeline([("encode", OneHotEncoder(handle_unknown="ignore", min_frequency=10))]), categorical),
    ], remainder="drop")
    treatment = treated_mask.astype(int).to_numpy()
    y = outcome.to_numpy(dtype="float64")
    order = np.argsort(frame["episode_start"].astype("int64").to_numpy(), kind="mergesort")
    fold_of = np.empty(len(frame), dtype=int)
    fold_of[order] = np.minimum((np.arange(len(frame)) * folds) // max(len(frame), 1), folds - 1)

    propensity = np.full(len(frame), np.nan)
    mu1 = np.full(len(frame), np.nan)
    mu0 = np.full(len(frame), np.nan)
    converged = True
    for fold in range(folds):
        train = fold_of != fold
        test = ~train
        if train.sum() < 40 or test.sum() < 10:
            continue
        if len(np.unique(treatment[train])) < 2:
            continue
        design_train = pre.fit_transform(design.loc[train])
        design_test = pre.transform(design.loc[test])
        # Unweighted: AIPW needs a calibrated P(treated | X), not a balanced score.
        propensity_model = LogisticRegression(max_iter=2000)
        propensity_model.fit(design_train, treatment[train])
        converged = converged and bool(np.all(propensity_model.n_iter_ < 2000))
        propensity[test] = propensity_model.predict_proba(design_test)[:, 1]
        for arm, sink in ((1, mu1), (0, mu0)):
            arm_rows = train & (treatment == arm)
            if arm_rows.sum() < 20 or len(np.unique(y[arm_rows])) < 2:
                continue
            outcome_model = LogisticRegression(max_iter=2000)
            outcome_model.fit(pre.transform(design.loc[arm_rows]), y[arm_rows])
            converged = converged and bool(np.all(outcome_model.n_iter_ < 2000))
            sink[test] = outcome_model.predict_proba(design_test)[:, 1]

    usable = ~np.isnan(propensity) & ~np.isnan(mu1) & ~np.isnan(mu0)
    if usable.sum() < 60:
        return None, "insufficient_crossfit_coverage"
    p_used = propensity[usable]
    t = treatment[usable]
    yy = y[usable]
    if t.sum() < 5 or (1 - t).sum() < 5:
        return None, "insufficient_arm_support_after_crossfit"
    # Common support: the propensity range the two arms actually share.
    lower = max(p_used[t == 1].min(), p_used[t == 0].min())
    upper = min(p_used[t == 1].max(), p_used[t == 0].max())
    in_support = (p_used >= lower) & (p_used <= upper)
    overlap = float(np.mean(in_support))
    clipped = np.clip(p_used, 0.01, 0.99)
    scores = (mu1[usable] - mu0[usable]
              + t * (yy - mu1[usable]) / clipped
              - (1 - t) * (yy - mu0[usable]) / (1 - clipped))
    ate = float(np.mean(scores))
    se = float(np.std(scores, ddof=1) / math.sqrt(len(scores)))
    smd = []
    for column in numeric:
        values = pd.to_numeric(frame.loc[usable, column], errors="coerce")
        a, b = values[t == 1], values[t == 0]
        pooled = math.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2) if len(a) > 1 and len(b) > 1 else np.nan
        smd.append({"covariate": column,
                    "standardised_mean_difference": float((a.mean() - b.mean()) / pooled) if pooled and pooled > 0 else np.nan})
    return {
        "ate": ate, "se": se, "ci_low": ate - 1.96 * se, "ci_high": ate + 1.96 * se,
        "n_used": int(usable.sum()), "n_treated": int(t.sum()), "n_control": int((1 - t).sum()),
        "overlap_share": overlap, "balance": smd,
        "treated_prevalence": float(t.mean()),
        "propensity_p01": float(np.quantile(p_used, 0.01)),
        "propensity_p99": float(np.quantile(p_used, 0.99)),
        "models_converged": converged,
    }, "estimated"


def ps3v22_causal_layer(facts):
    if not PS3V21_CONFIG["ENABLE_CAUSAL_LAYER"]:
        return pd.DataFrame(), pd.DataFrame(), {"status": "disabled", "detail": "ENABLE_CAUSAL_LAYER is false."}
    if not PS3V21_SKLEARN_READY:
        return pd.DataFrame(), pd.DataFrame(), {"status": "skipped_dependency", "detail": "scikit-learn unavailable."}
    horizon = int(PS3V21_CONFIG["CAUSAL_REPEAT_HORIZON_DAYS"])
    outcome, days_to_next = ps3v22_repeat_outcome(facts, horizon)
    frame = facts.copy()
    frame["_repeat_within_horizon"] = outcome
    frame["_days_to_next_episode"] = days_to_next
    attributed = frame.loc[frame["component_attribution"].notna() & frame["episode_start"].notna()].copy()
    if len(attributed) < PS3V21_CONFIG["CAUSAL_MIN_TREATED"] + PS3V21_CONFIG["CAUSAL_MIN_CONTROL"]:
        return pd.DataFrame(), pd.DataFrame(), {"status": "skipped_insufficient_attributed_episodes",
                                                "detail": f"{len(attributed)} component-attributed episodes."}
    numeric, categorical = ps3v22_causal_confounders(attributed)
    estimates, balances = [], []
    counts = attributed["component_attribution"].value_counts()
    for component, treated_n in counts.items():
        control_n = len(attributed) - treated_n
        if treated_n < PS3V21_CONFIG["CAUSAL_MIN_TREATED"] or control_n < PS3V21_CONFIG["CAUSAL_MIN_CONTROL"]:
            continue
        treated_mask = attributed["component_attribution"] == component
        result, status = ps3v22_aipw_effect(
            attributed, treated_mask, attributed["_repeat_within_horizon"],
            numeric, categorical, int(PS3V21_CONFIG["CAUSAL_FOLDS"]),
        )
        if result is None:
            estimates.append({"treatment_component": component, "status": status,
                              "outcome": f"repeat_oos_episode_within_{horizon}d"})
            continue
        estimates.append({
            "treatment_component": component,
            "outcome": f"repeat_oos_episode_within_{horizon}d",
            "estimator": "cross_fitted_aipw_time_folds",
            "average_treatment_effect": result["ate"],
            "standard_error": result["se"],
            "ci_low_95": result["ci_low"], "ci_high_95": result["ci_high"],
            "significant_95": bool(result["ci_low"] > 0 or result["ci_high"] < 0),
            "episodes_used": result["n_used"], "treated_episodes": result["n_treated"],
            "control_episodes": result["n_control"], "overlap_share": result["overlap_share"],
            "treated_prevalence": result["treated_prevalence"],
            "propensity_p01": result["propensity_p01"], "propensity_p99": result["propensity_p99"],
            "models_converged": result["models_converged"],
            "status": ("estimated" if (result["overlap_share"] >= 0.80 and result["models_converged"])
                       else "estimated_weak_overlap" if result["overlap_share"] < 0.80
                       else "estimated_model_not_converged"),
            "interpretation": ("Observational, not experimental. Confounding by unmeasured maintenance "
                               "policy is not ruled out. Read as association adjusted for the recorded "
                               "pre-event history, never as proof of causation."),
        })
        for row in result["balance"]:
            balances.append({"treatment_component": component, **row,
                             "balance_status": "acceptable" if abs(row["standardised_mean_difference"] or 0) <= 0.10 else "imbalanced"})
    if not estimates:
        return pd.DataFrame(), pd.DataFrame(), {"status": "no_component_met_support_floor",
                                                "detail": f"No component had >= {PS3V21_CONFIG['CAUSAL_MIN_TREATED']} treated episodes."}
    frame = ps3v22_holm_adjust(pd.DataFrame(estimates))
    return frame, pd.DataFrame(balances), {"status": "completed", "detail": f"{len(frame)} component contrasts."}


def ps3v22_holm_adjust(frame):
    """Holm-Bonferroni across the component contrasts.

    Every component is a separate hypothesis test on the same episodes. With a
    dozen components, one or two will clear a naive 95% bound by chance alone,
    and a dashboard that shows only the raw flag will present that noise as a
    finding. significant_95_holm is the column to trust.
    """
    if "average_treatment_effect" not in frame.columns:
        return frame
    valid = frame["standard_error"].notna() & (frame["standard_error"] > 0)
    z = pd.Series(np.nan, index=frame.index, dtype="float64")
    z.loc[valid] = frame.loc[valid, "average_treatment_effect"] / frame.loc[valid, "standard_error"]
    frame["p_value_two_sided"] = z.abs().map(lambda v: math.erfc(v / math.sqrt(2)) if v == v else np.nan)
    tested = frame.loc[frame["p_value_two_sided"].notna()].sort_values("p_value_two_sided")
    m = len(tested)
    adjusted, running = {}, 0.0
    for rank, (index, row) in enumerate(tested.iterrows()):
        value = min(1.0, (m - rank) * row["p_value_two_sided"])
        running = max(running, value)
        adjusted[index] = running
    frame["p_value_holm"] = pd.Series(adjusted)
    frame["significant_95_holm"] = frame["p_value_holm"] < 0.05
    frame["multiplicity_note"] = f"Holm-Bonferroni across {m} component contrasts on the same episode set."
    return frame


In [7]:
# 6. Dashboard tables and immutable S3 Parquet publication.

def ps3v21_add_delivery_metadata(facts, run_id):
    frame = facts.copy()
    as_of = pd.Timestamp(PS3V21_CONFIG["DATA_AS_OF_DATE"])
    lag_days = max(0, (pd.Timestamp(ps3v21_now_utc().replace(tzinfo=None).date()) - as_of).days)
    frame["run_id"] = run_id
    frame["computed_at_utc"] = ps3v21_now_utc().isoformat()
    frame["data_as_of_date"] = PS3V21_CONFIG["DATA_AS_OF_DATE"]
    frame["data_freshness_days"] = lag_days
    frame["is_current_operational_score"] = False
    frame["freshness_status"] = "historical_backtest_right_censored" if lag_days > PS3V21_CONFIG["PS1_HORIZON_DAYS"] else "near_current_governed_snapshot"
    frame["right_censored_tail_days"] = lag_days
    frame["chargeability_policy"] = "not_read_not_used_for_oos_discovery_or_modelling"
    frame["shap_interpretation"] = "model_driver_only_not_physical_causation"
    return frame


def ps3v21_dashboard_tables(facts, scorecards, feature_importance, explanations, source_audit, evidence_audit, run_id):
    facts = ps3v21_add_delivery_metadata(facts, run_id)
    # Confirmed evidence takes precedence. Quality-gated model outputs remain
    # clearly marked as estimates and cannot overwrite a confirmation.
    if "predicted_root_cause" in facts:
        facts["dashboard_root_cause_domain"] = facts["confirmed_root_cause_domain"].fillna(facts["predicted_root_cause"])
        facts["dashboard_root_cause_status"] = np.where(facts["confirmed_root_cause_domain"].notna(), "confirmed_evidence_and_taxonomy", "model_estimate_not_confirmed")
    else:
        facts["dashboard_root_cause_domain"] = facts["confirmed_root_cause_domain"]
        facts["dashboard_root_cause_status"] = facts["root_cause_status"]
    if "predicted_severity" in facts:
        facts["dashboard_severity"] = facts["observed_severity_label"].fillna(facts["predicted_severity"])
        facts["dashboard_severity_status"] = np.where(facts["observed_severity_label"].notna(), "observed_linked_label", "model_estimate_not_observed")
    else:
        facts["dashboard_severity"] = facts["observed_severity_label"]
        facts["dashboard_severity_status"] = facts["severity_status"]
    device_summary = facts.groupby(["device_id", "mars_device_category"], dropna=False).agg(
        oos_episode_count=("oos_episode_id", "nunique"),
        first_oos_episode_start=("episode_start", "min"),
        latest_oos_episode_start=("episode_start", "max"),
        latest_dashboard_severity=("dashboard_severity", "last"),
        latest_dashboard_root_cause_domain=("dashboard_root_cause_domain", "last"),
        confirmed_root_cause_episode_count=("root_cause_status", lambda s: int((s == "confirmed_evidence_and_taxonomy").sum())),
        observed_severity_episode_count=("severity_status", lambda s: int((s == "observed_linked_label").sum())),
    ).reset_index()
    # V26: the loader refuses a table whose declared key is null, and these four
    # columns are the natural key. dashboard_root_cause_domain is entirely null
    # whenever the PS3 evidence lane is unconfigured, which is the normal state
    # today. Sentinel-filling keeps the key valid and, unlike dropping the column
    # from the key, it stays correct once real root causes arrive.
    _summary_keys = ["mars_device_category", "component_attribution",
                     "dashboard_root_cause_domain", "dashboard_severity"]
    for _k in _summary_keys:
        if _k not in facts.columns:
            facts[_k] = pd.NA
        facts[_k] = facts[_k].astype("string").fillna("(unlabelled)")
    component_summary = facts.groupby(_summary_keys, dropna=False).agg(
        oos_episode_count=("oos_episode_id", "nunique"),
        device_count=("device_id", "nunique"),
        confirmed_root_cause_count=("root_cause_status", lambda s: int((s == "confirmed_evidence_and_taxonomy").sum())),
    ).reset_index()
    label_maturity = facts.groupby("mars_device_category", dropna=False).agg(
        oos_episode_count=("oos_episode_id", "nunique"),
        observed_severity_count=("observed_severity_label", lambda s: int(s.notna().sum())),
        confirmed_root_cause_count=("confirmed_root_cause_label", lambda s: int(s.notna().sum())),
        candidate_root_cause_count=("candidate_root_cause_raw", lambda s: int(s.notna().sum())),
        component_attribution_count=("component_attribution", lambda s: int(s.notna().sum())),
    ).reset_index()
    label_maturity["severity_coverage"] = label_maturity["observed_severity_count"] / label_maturity["oos_episode_count"].replace(0, np.nan)
    label_maturity["confirmed_root_cause_coverage"] = label_maturity["confirmed_root_cause_count"] / label_maturity["oos_episode_count"].replace(0, np.nan)
    # V26: sentinel-fill the explainability key columns at the PUBLISH boundary,
    # not at each producer. Placeholder rows are emitted from more than one path
    # -- the per-scope trainer and the entry-point fallback -- and a null key
    # makes the whole table unloadable, so this is enforced once, where it can
    # be guaranteed, rather than in every place that might create a row.
    if isinstance(explanations, pd.DataFrame) and not explanations.empty:
        for _key in ("target", "model_output", "oos_episode_id", "feature"):
            if _key not in explanations.columns:
                explanations[_key] = pd.NA
            explanations[_key] = explanations[_key].astype("string").fillna("(not_computed)")
        # The key must also be unique, or the loader refuses the table.
        explanations = explanations.drop_duplicates(
            subset=["target", "model_output", "oos_episode_id", "feature"], keep="first")

    source_frame = pd.DataFrame(source_audit)
    evidence_frame = pd.DataFrame(evidence_audit)
    # --- V22: commanded split, so the dashboard can separate planned work from
    # failure without PS3 diverging from PS2 on what an OOS episode is --------
    for frame_name, frame_obj in (("device", device_summary),):
        pass
    commanded = facts.groupby("mars_device_category", dropna=False)["contains_commanded_oos_signal"].agg(
        oos_episodes="size", commanded_signal_episodes="sum").reset_index()
    commanded["failure_only_episodes"] = (commanded["oos_episodes"] - commanded["commanded_signal_episodes"]).clip(lower=0)
    commanded["basis"] = "commanded OOS is retained in the spine so PS3 reconciles with PS1 and PS2; use failure_only_episodes to exclude planned work"

    horizon = int(PS3V21_CONFIG["CAUSAL_REPEAT_HORIZON_DAYS"])
    causal_effects, causal_balance, causal_status = ps3v22_causal_layer(facts)
    ps3v21_stage("causal_layer", causal_status["status"], causal_status.get("detail"))

    return {
        "ps3_device_episode_fact": facts,
        "ps3_device_day": ps3v22_device_day(facts),
        "ps3_facility_rollup": ps3v22_facility_rollup(facts),
        "ps3_device_reliability": ps3v22_device_reliability(facts),
        "ps3_serial_reliability": ps3v22_serial_reliability(facts),
        "ps3_repeat_interval": ps3v22_repeat_interval(facts, horizon),
        "ps3_commanded_split": commanded,
        "ps3_causal_effects": causal_effects if not causal_effects.empty else pd.DataFrame([{"status": causal_status["status"], "detail": causal_status.get("detail")}]),
        "ps3_causal_balance": causal_balance if not causal_balance.empty else pd.DataFrame([{"status": causal_status["status"]}]),
        "ps3_device_summary": device_summary,
        "ps3_component_summary": component_summary,
        "ps3_label_maturity": label_maturity,
        "ps3_model_scorecard": scorecards if not scorecards.empty else pd.DataFrame(columns=["target", "candidate_model", "quality_gate"]),
        "ps3_model_feature_importance": feature_importance if not feature_importance.empty else pd.DataFrame(columns=["target", "model", "feature", "importance"]),
        "ps3_prediction_explainability": explanations if not explanations.empty else pd.DataFrame(columns=["target", "model_output", "explanation_status", "explanation_note"]),
        "ps3_oos_source_audit": source_frame,
        "ps3_root_cause_evidence_audit": evidence_frame,
    }


def ps3v21_write_local_frame(frame, table_name, run_id):
    local_root = Path(PS3V21_CONFIG["LOCAL_OUTPUT_ROOT"]) / run_id / table_name
    local_root.mkdir(parents=True, exist_ok=True)
    if PS3V21_PYARROW_READY:
        path = local_root / "part-00000.parquet"
        frame.to_parquet(path, index=False)
        return str(path)
    path = local_root / "part-00000.csv"
    frame.to_csv(path, index=False)
    return str(path)


def ps3v21_write_pandas_parquet_s3(frame, table_name, run_id):
    path = ps3v21_output_path(table_name, run_id) + "/part-00000.parquet"
    if not PS3V21_CONFIG["PUBLISH_TO_S3"]:
        return {"table": table_name, "status": "publication_disabled", "path": path, "rows": len(frame)}
    if not (PS3V21_BOTO3_READY and PS3V21_PYARROW_READY):
        spark_session = globals().get("spark")
        if spark_session is not None:
            try:
                # Fallback for lean Studio images: preserve the Parquet/S3
                # contract through the active Spark writer without requiring
                # boto3 or pyarrow in the Python environment.
                safe = frame.astype(object).where(pd.notna(frame), None)
                spark_frame = spark_session.createDataFrame(safe)
                spark_frame.write.mode("overwrite").parquet(ps3v21_s3a(path.rsplit("/part-00000.parquet", 1)[0]))
                return {"table": table_name, "status": "published_via_spark_fallback", "path": path.rsplit("/part-00000.parquet", 1)[0], "rows": len(frame)}
            except Exception as exc:
                return {"table": table_name, "status": "publish_failed", "path": path, "rows": len(frame), "detail": ps3v21_error_detail(exc)}
        return {"table": table_name, "status": "not_published_missing_boto3_or_pyarrow", "path": path, "rows": len(frame), "detail": "No pandas Parquet writer and no active Spark writer are available."}
    try:
        bucket, key = ps3v21_parse_s3(path)
        buffer = BytesIO()
        table = pa.Table.from_pandas(frame, preserve_index=False)
        pq.write_table(table, buffer, compression="snappy")
        boto3.client("s3").put_object(Bucket=bucket, Key=key, Body=buffer.getvalue(), ContentType="application/octet-stream")
        return {"table": table_name, "status": "published", "path": path, "rows": len(frame)}
    except Exception as exc:
        return {"table": table_name, "status": "publish_failed", "path": path, "rows": len(frame), "detail": ps3v21_error_detail(exc)}


def ps3v21_write_spark_parquet(frame, table_name, run_id):
    path = ps3v21_output_path(table_name, run_id)
    if not PS3V21_CONFIG["PUBLISH_TO_S3"]:
        return {"table": table_name, "status": "publication_disabled", "path": path}
    try:
        # This is an explicit immutable run partition. `overwrite` can affect
        # only this exact new run-id path; it never clears existing runs.
        frame.write.mode("overwrite").parquet(ps3v21_s3a(path))
        return {"table": table_name, "status": "published", "path": path}
    except Exception as exc:
        return {"table": table_name, "status": "publish_failed", "path": path, "detail": ps3v21_error_detail(exc)}


def ps3v21_write_manifest(records, summary, run_id):
    manifest = {
        "run_id": run_id,
        "revision": PS3V21_CONFIG["PS3_REVISION"],
        "computed_at_utc": ps3v21_now_utc().isoformat(),
        "data_as_of_date": PS3V21_CONFIG["DATA_AS_OF_DATE"],
        "publication_policy": "immutable run partition; no broad cleanup",
        "run_mode": PS3V21_CONFIG["RUN_MODE"],
        "tables": records,
        "summary": summary,
    }
    local = Path(PS3V21_CONFIG["LOCAL_OUTPUT_ROOT"]) / run_id
    local.mkdir(parents=True, exist_ok=True)
    (local / "ps3_run_summary.json").write_text(json.dumps(manifest, indent=2, default=str))
    manifest_path = ps3v21_output_path("ps3_run_control", run_id) + "/manifest.json"
    if PS3V21_CONFIG["PUBLISH_TO_S3"] and PS3V21_BOTO3_READY:
        try:
            bucket, key = ps3v21_parse_s3(manifest_path)
            boto3.client("s3").put_object(Bucket=bucket, Key=key, Body=json.dumps(manifest, default=str).encode("utf-8"), ContentType="application/json")
            manifest["manifest_s3_path"] = manifest_path
        except Exception as exc:
            manifest["manifest_publish_error"] = ps3v21_error_detail(exc)
    # --- V22: the loader trigger. Written LAST and only for a PRODUCTION run
    # -- the RUN_MODE test below was MISSING until 08-Aug-2026, so a REPLAY
    # run emitted this marker too. It landed under the replay prefix, so the
    # isolation held by accident: any consumer pointed at ps3_replay_outputs
    # saw a completion marker indistinguishable from a production one. That
    # is exactly what happened to cubic-mars-ps3-v25-loader.
    # whose tables all published, so a partial run can never be picked up. ---
    if (PS3V21_CONFIG["RUN_MODE"] == "PRODUCTION"
        and PS3V21_CONFIG["EMIT_RUN_COMPLETE"] and PS3V21_CONFIG["PUBLISH_TO_S3"]
        and PS3V21_BOTO3_READY):
        published = [r for r in records if r.get("status", "").startswith("published")]
        if published and len(published) == len(records):
            complete_path = (f"{PS3V21_CONFIG['S3_OUTPUT_PREFIX']}/_runs/"
                             f"computed_date={PS3V21_CONFIG['DATA_AS_OF_DATE']}/run_id={run_id}/run_complete.json")
            try:
                bucket, key = ps3v21_parse_s3(complete_path)
                boto3.client("s3").put_object(
                    Bucket=bucket, Key=key,
                    Body=json.dumps({"run_id": run_id, "computed_date": PS3V21_CONFIG["DATA_AS_OF_DATE"],
                                     "revision": PS3V21_CONFIG["PS3_REVISION"], "tables": len(records),
                                     "run_mode": PS3V21_CONFIG["RUN_MODE"]}, default=str).encode("utf-8"),
                    ContentType="application/json")
                manifest["run_complete_path"] = complete_path
            except Exception as exc:
                manifest["run_complete_error"] = ps3v21_error_detail(exc)
        else:
            manifest["run_complete_withheld"] = (
                f"{len(published)} of {len(records)} tables published; the loader trigger is "
                "written only for a complete run.")
    return manifest


# ---------------------------------------------------------------------------
# V22 ADDITION: the dashboard and Device 360 tables V21 did not produce.
# Every one is derived from the OOS spine PS3 already builds. None of them
# introduces a new source, so none can disagree with PS1 or PS2 about what an
# out-of-service episode is.
# ---------------------------------------------------------------------------
def ps3v22_optional(frame, column, dtype="string"):
    if column in frame.columns:
        return frame[column]
    return pd.Series(pd.NA, index=frame.index, dtype=dtype)


def ps3v22_device_day(facts):
    """Device x calendar-day grain, keyed to the same date PS1 and PS2 use.

    PS1 keys on transit_day and PS2 on event_date; both are plain calendar
    dates. Publishing event_date here is what lets Device 360 draw one
    timeline instead of three cards that cannot be lined up.
    """
    frame = facts.copy()
    frame["event_date"] = frame["episode_start"].dt.normalize()
    grouped = frame.groupby(["device_id", "mars_device_category", "event_date"], dropna=False)
    table = grouped.agg(
        oos_episode_starts=("oos_episode_id", "nunique"),
        oos_set_events=("oos_set_event_count", "sum"),
        set_signal_span_minutes=("set_signal_span_minutes", "sum"),
        commanded_signal_episodes=("contains_commanded_oos_signal", "sum"),
        observed_severity_episodes=("severity_status", lambda s: int((s == "observed_linked_label").sum())),
        confirmed_root_cause_episodes=("root_cause_status", lambda s: int((s == "confirmed_evidence_and_taxonomy").sum())),
    ).reset_index()
    table["failure_only_episode_starts"] = (
        table["oos_episode_starts"] - table["commanded_signal_episodes"]
    ).clip(lower=0)
    table["grain"] = "device_day"
    return table


def ps3v22_facility_rollup(facts):
    facility = ps3v22_optional(facts, "facility_id")
    if facility.isna().all():
        return pd.DataFrame([{ "facility_id": pd.NA, "status": "unavailable",
                               "detail": "FACILITY_ID was not present in the OOS source projection."}])
    frame = facts.copy()
    frame["facility_id"] = facility
    frame["event_date"] = frame["episode_start"].dt.normalize()
    table = frame.groupby(["facility_id", "mars_device_category"], dropna=False).agg(
        devices=("device_id", "nunique"),
        oos_episodes=("oos_episode_id", "nunique"),
        first_episode=("episode_start", "min"),
        latest_episode=("episode_start", "max"),
        active_days=("event_date", "nunique"),
        commanded_signal_episodes=("contains_commanded_oos_signal", "sum"),
        confirmed_root_cause_episodes=("root_cause_status", lambda s: int((s == "confirmed_evidence_and_taxonomy").sum())),
    ).reset_index()
    table["episodes_per_device"] = table["oos_episodes"] / table["devices"].replace(0, np.nan)
    table["failure_only_episodes"] = (table["oos_episodes"] - table["commanded_signal_episodes"]).clip(lower=0)
    return table.sort_values("oos_episodes", ascending=False).reset_index(drop=True)


def ps3v22_device_reliability(facts):
    """Restores the device reliability pack the deployed Device 360 reads."""
    frame = facts.sort_values(["device_id", "episode_start"], kind="mergesort").copy()
    intervals = frame.groupby("device_id", sort=False)["episode_start"].diff().dt.total_seconds() / 3600.0
    frame["_interval_hours"] = intervals
    critical = frame["dashboard_severity"].astype("string").str.upper().isin(["CRITICAL", "SEV1", "HIGH"]) \
        if "dashboard_severity" in frame.columns else pd.Series(False, index=frame.index)
    frame["_critical"] = critical.fillna(False)
    table = frame.groupby(["device_id", "mars_device_category"], dropna=False).agg(
        oos_episode_count=("oos_episode_id", "nunique"),
        oos_set_event_count=("oos_set_event_count", "sum"),
        critical_episodes=("_critical", "sum"),
        first_episode_at=("episode_start", "min"),
        latest_episode_at=("episode_start", "max"),
        mean_interval_hours=("_interval_hours", "mean"),
        median_interval_hours=("_interval_hours", "median"),
        confirmed_root_cause_episodes=("root_cause_status", lambda s: int((s == "confirmed_evidence_and_taxonomy").sum())),
        observed_severity_episodes=("severity_status", lambda s: int((s == "observed_linked_label").sum())),
        commanded_signal_episodes=("contains_commanded_oos_signal", "sum"),
    ).reset_index()
    table["critical_rate"] = table["critical_episodes"] / table["oos_episode_count"].replace(0, np.nan)
    table["failure_only_episode_count"] = (
        table["oos_episode_count"] - table["commanded_signal_episodes"]
    ).clip(lower=0)
    # Bands are ordinal and observed, not a model score. They rank devices by
    # how often they go out of service and how quickly they come back.
    rate = table["oos_episode_count"].rank(pct=True)
    table["reliability_risk_band"] = np.where(rate >= 0.95, "WORST_5_PCT",
                                     np.where(rate >= 0.80, "WORST_20_PCT",
                                     np.where(rate >= 0.50, "ABOVE_MEDIAN", "BELOW_MEDIAN")))
    table["band_basis"] = "observed_episode_count_percentile_not_a_model_score"
    return table.sort_values("oos_episode_count", ascending=False).reset_index(drop=True)


def ps3v22_serial_reliability(facts):
    serial = ps3v22_optional(facts, "component_serial_nbr")
    if serial.isna().all():
        return pd.DataFrame([{ "component_serial_nbr": pd.NA, "status": "unavailable",
                               "detail": "COMPONENT_SERIAL_NBR was not present in the OOS source projection."}])
    frame = facts.copy()
    frame["component_serial_nbr"] = serial
    frame = frame.loc[frame["component_serial_nbr"].notna()]
    if frame.empty:
        return pd.DataFrame([{ "component_serial_nbr": pd.NA, "status": "no_serial_rows"}])
    table = frame.groupby(["component_serial_nbr", "device_id", "mars_device_category"], dropna=False).agg(
        oos_episode_count=("oos_episode_id", "nunique"),
        first_episode_at=("episode_start", "min"),
        latest_episode_at=("episode_start", "max"),
        component_attributions=("component_attribution", lambda s: int(s.notna().sum())),
        confirmed_root_cause_episodes=("root_cause_status", lambda s: int((s == "confirmed_evidence_and_taxonomy").sum())),
    ).reset_index()
    span_days = (table["latest_episode_at"] - table["first_episode_at"]).dt.total_seconds() / 86400.0
    table["observed_span_days"] = span_days
    table["episodes_per_100_observed_days"] = np.where(
        span_days > 0, table["oos_episode_count"] * 100.0 / span_days, np.nan)
    return table.sort_values("oos_episode_count", ascending=False).reset_index(drop=True)


def ps3v22_repeat_interval(facts, horizon_days):
    """Observed time to the next OOS episode after a component-attributed one.

    Deliberately NOT called repair effectiveness. Without the maintenance
    ledger we do not know a repair happened, only that an episode was
    attributed to a component and what followed it.
    """
    frame = facts.loc[facts["component_attribution"].notna()].copy()
    if frame.empty:
        return pd.DataFrame([{ "component_attribution": pd.NA, "status": "no_component_attributed_episodes"}])
    _, days_to_next = ps3v22_repeat_outcome(facts, horizon_days)
    frame["days_to_next_episode"] = days_to_next.loc[frame.index]
    table = frame.groupby(["component_attribution", "mars_device_category"], dropna=False).agg(
        attributed_episodes=("oos_episode_id", "nunique"),
        devices=("device_id", "nunique"),
        episodes_with_a_next=("days_to_next_episode", lambda s: int(s.notna().sum())),
        median_days_to_next=("days_to_next_episode", "median"),
        p25_days_to_next=("days_to_next_episode", lambda s: s.quantile(0.25)),
        mean_days_to_next=("days_to_next_episode", "mean"),
    ).reset_index()
    table["repeat_within_horizon_rate"] = table.apply(
        lambda r: np.nan if not r["episodes_with_a_next"] else None, axis=1)
    repeat = frame.assign(_hit=(frame["days_to_next_episode"] <= horizon_days).astype("float64")) \
                  .groupby(["component_attribution", "mars_device_category"], dropna=False)["_hit"].mean() \
                  .reset_index().rename(columns={"_hit": "repeat_rate_within_horizon"})
    table = table.drop(columns=["repeat_within_horizon_rate"]).merge(
        repeat, on=["component_attribution", "mars_device_category"], how="left")
    table["horizon_days"] = horizon_days
    table["basis"] = "observed_inter_episode_interval_not_a_verified_repair"
    return table.sort_values("attributed_episodes", ascending=False).reset_index(drop=True)


def ps3v22_run_status(records, run_id, summary):
    """The /ps3/status equivalent: one row per table, one coherent run id.

    PS2 has this and it is the only cheap way to tell a half-loaded dashboard
    from a healthy one.
    """
    rows = []
    for record in records:
        rows.append({
            "run_id": run_id,
            "revision": PS3V21_CONFIG["PS3_REVISION"],
            "computed_date": PS3V21_CONFIG["DATA_AS_OF_DATE"],
            "table_name": record.get("table"),
            "publish_status": record.get("status"),
            "rows": record.get("rows"),
            "path": record.get("path"),
            "run_mode": PS3V21_CONFIG["RUN_MODE"],
            "data_as_of_date": PS3V21_CONFIG["DATA_AS_OF_DATE"],
            "is_current_operational_score": False,
            "computed_at_utc": ps3v21_now_utc().isoformat(),
        })
    frame = pd.DataFrame(rows)
    if not frame.empty:
        frame["run_is_coherent"] = frame["run_id"].nunique() == 1
        frame["tables_published"] = int((frame["publish_status"] == "published").sum())
        frame["tables_total"] = len(frame)
    return frame


In [8]:
# 7. One entry point — it never raises a nested exception.

def ps3v21_publish_control_only(run_id, source_audit, status, detail):
    control = pd.DataFrame([{
        "run_id": run_id,
        "revision": PS3V21_CONFIG["PS3_REVISION"],
        "status": status,
        "detail": detail,
        "data_as_of_date": PS3V21_CONFIG["DATA_AS_OF_DATE"],
        "is_current_operational_score": False,
        "at_utc": ps3v21_now_utc().isoformat(),
    }])
    tables = {"ps3_run_control": control, "ps3_oos_source_audit": pd.DataFrame(source_audit)}
    records = []
    for name, frame in tables.items():
        ps3v21_write_local_frame(frame, name, run_id)
        records.append(ps3v21_write_pandas_parquet_s3(frame, name, run_id))
    manifest = ps3v21_write_manifest(records, {"status": status, "detail": detail}, run_id)
    return {"status": status, "run_id": run_id, "tables": tables, "publication": records, "manifest": manifest}


def run_ps3_v21():
    """Run PS3 V21 safely. It returns a result object for every known condition."""
    PS3V21_STATE["run_id"] = ps3v21_run_id()
    PS3V21_STATE["stage_audit"] = []
    PS3V21_STATE["publication"] = []
    PS3V21_STATE["models"] = {}
    run_id = PS3V21_STATE["run_id"]
    try:
        if not PS3V21_PANDAS_READY:
            return {"status": "not_ready_dependencies", "run_id": run_id, "detail": globals().get("PS3V21_IMPORT_ERROR")}
        preflight = run_ps3_v21_preflight()
        ps3v21_stage("preflight", "completed", rows=len(preflight))
        source = ps3v21_load_oos_spine()
        ps3v21_stage("oos_source", source["status"], source.get("detail"), mode=source.get("mode"))
        if source["status"] != "loaded":
            result = ps3v21_publish_control_only(run_id, source.get("audit", []), "not_ready_oos_source", source.get("detail", "No verified PS1 OOS source."))
            result["preflight"] = preflight
            return result

        # Persist the bounded direct-Silver projection first. This is an input
        # audit artifact only; it is not an RDS trigger or a Gold OOS source.
        bootstrap_record = None
        if source.get("event_facts_spark") is not None and PS3V21_CONFIG["WRITE_BOUNDED_SOURCE_EXPORT"]:
            bootstrap_path = (
                f"{PS3V21_CONFIG['S3_BOOTSTRAP_PREFIX']}/as_of_date={PS3V21_CONFIG['DATA_AS_OF_DATE']}/run_id={run_id}"
            )
            try:
                source["event_facts_spark"].write.mode("overwrite").partitionBy("mars_device_category").parquet(ps3v21_s3a(bootstrap_path))
                bootstrap_record = {"table": "ps3_ps1_hardware_oos_set_event_export", "status": "published", "path": bootstrap_path}
            except Exception as exc:
                bootstrap_record = {"table": "ps3_ps1_hardware_oos_set_event_export", "status": "publish_failed", "path": bootstrap_path, "detail": ps3v21_error_detail(exc)}
        episodes, collection = ps3v21_collect_episodes(source)
        ps3v21_stage("episode_collection", collection["status"], collection.get("detail"), rows=collection.get("rows"))
        if collection["status"] != "loaded":
            records = []
            if source.get("episodes_spark") is not None:
                records.append(ps3v21_write_spark_parquet(source["episodes_spark"], "ps3_device_episode_fact_raw", run_id))
            control_result = ps3v21_publish_control_only(run_id, source.get("audit", []), "facts_published_local_enrichment_deferred", collection.get("detail"))
            control_result["publication"] = records + control_result["publication"]
            control_result["preflight"] = preflight
            control_result["bootstrap"] = bootstrap_record
            return control_result

        spark_session = globals().get("spark")
        facts, evidence_audit, taxonomy = ps3v21_enrich_episodes(episodes, spark_session)
        ps3v21_stage("evidence_factory", "completed", evidence_sources=len(evidence_audit), taxonomy_rows=len(taxonomy))
        # V24: profile the candidate source label columns, then promote source
        # severity only if it survives the non-triviality gate.
        source_profile = ps3v22_source_column_profile(facts)
        facts, severity_promotion = ps3v22_promote_source_severity(facts, source_profile)
        ps3v21_stage("source_label_profile", severity_promotion["status"], severity_promotion.get("detail"))

        facts = ps3v21_add_pre_event_features(facts)
        facts, ps1_feature_audit = ps3v21_add_feature_snapshot(facts, spark_session, PS3V21_CONFIG["PS1_FEATURE_EXPORT"], "ps1_feature")
        facts, ps2_feature_audit = ps3v21_add_feature_snapshot(facts, spark_session, PS3V21_CONFIG["PS2_FEATURE_EXPORT"], "ps2_feature")
        evidence_audit.extend([ps1_feature_audit, ps2_feature_audit])
        ps3v21_stage("pre_event_features", "completed")

        model_runs = []
        scorecards, importances, explanations = [], [], []
        # Severity is observed PS3 evidence; root cause is only a confirmed
        # taxonomy/domain target. Component is an attribution target, never a
        # substitute for physical root cause.
        target_specs = [
            ("observed_severity_label", "severity"),
            ("confirmed_root_cause_domain", "root_cause"),
            ("component_attribution", "component"),
        ]
        # V22: trained per device category. A pooled gate can pass on one
        # fleet's strength and ship a model that is useless for another.
        for target, model_key in target_specs:
            outcome = ps3v22_train_target_by_scope(facts, target, model_key)
            model_runs.extend(outcome["runs"])
            if not outcome["scorecard"].empty:
                scorecards.append(outcome["scorecard"])
            if not outcome["feature_importance"].empty:
                importances.append(outcome["feature_importance"])
            if not outcome["explainability"].empty:
                explanations.append(outcome["explainability"])
            else:
                explanations.append(pd.DataFrame([{
                    "target": target,
                    "model_output": model_key,
                    # V26: sentinels, not nulls. oos_episode_id and feature are part
                    # of this table's natural key, and a placeholder row with null
                    # keys makes the whole table unloadable.
                    "oos_episode_id": "(not_computed)",
                    "device_id": pd.NA,
                    "mars_device_category": pd.NA,
                    "predicted_label": pd.NA,
                    "feature": "(not_computed)",
                    "shap_value": np.nan,
                    "abs_shap_value": np.nan,
                    "explanation_status": "not_computed_model_not_publishable",
                    "explanation_note": "SHAP is calculated only for a quality-gated model and is never physical causation.",
                }]))
            # Model-owned columns only. Observed and confirmed evidence stays
            # authoritative and is never overwritten by an estimate.
            for column, values in outcome["scored_columns"].items():
                facts[column] = values
        PS3V21_STATE["models"] = model_runs
        ps3v21_stage("models", "completed", model_runs=model_runs)
        combined_scorecards = pd.concat(scorecards, ignore_index=True, sort=False) if scorecards else pd.DataFrame()
        combined_importance = pd.concat(importances, ignore_index=True, sort=False) if importances else pd.DataFrame()
        combined_explanations = pd.concat(explanations, ignore_index=True, sort=False) if explanations else pd.DataFrame()
        tables = ps3v21_dashboard_tables(facts, combined_scorecards, combined_importance, combined_explanations, source.get("audit", []), evidence_audit, run_id)
        tables["ps3_run_stage_audit"] = pd.DataFrame(PS3V21_STATE["stage_audit"])
        tables["ps3_source_column_profile"] = source_profile

        records = []
        if bootstrap_record:
            records.append(bootstrap_record)
        for name, frame in tables.items():
            local_path = ps3v21_write_local_frame(frame, name, run_id)
            record = ps3v21_write_pandas_parquet_s3(frame, name, run_id)
            record["local_path"] = local_path
            records.append(record)
        summary = {
            "status": "completed",
            "source_mode": source.get("mode"),
            "oos_episode_rows": int(len(facts)),
            "model_runs": model_runs,
            "confirmed_root_cause_rows": int(facts["confirmed_root_cause_label"].notna().sum()),
            "observed_severity_rows": int(facts["observed_severity_label"].notna().sum()),
            "validator_oos_rows": int((facts["mars_device_category"] == "VALIDATOR").sum()),
            "commanded_signal_rows": int(facts["contains_commanded_oos_signal"].sum()),
            "failure_only_rows": int((~facts["contains_commanded_oos_signal"].astype(bool)).sum()),
            "episode_scope_status": str(facts["episode_scope_status"].iloc[0]) if "episode_scope_status" in facts and len(facts) else "unknown",
            "run_mode": PS3V21_CONFIG["RUN_MODE"],
            "historical_snapshot": True,
            "data_as_of_date": PS3V21_CONFIG["DATA_AS_OF_DATE"],
        }
        # V22: the /ps3/status equivalent, written after every other table so it
        # can report their publish outcomes and prove one coherent run id.
        status_frame = ps3v22_run_status(records, run_id, summary)
        status_record = ps3v21_write_pandas_parquet_s3(status_frame, "ps3_run_status", run_id)
        status_record["local_path"] = ps3v21_write_local_frame(status_frame, "ps3_run_status", run_id)
        records.append(status_record)
        tables["ps3_run_status"] = status_frame

        manifest = ps3v21_write_manifest(records, summary, run_id)
        ps3v21_stage("publication", "completed", published=sum(r.get("status") == "published" for r in records), total=len(records))
        return {"status": "completed", "run_id": run_id, "preflight": preflight, "summary": summary, "tables": tables, "publication": records, "manifest": manifest, "source_audit": pd.DataFrame(source.get("audit", [])), "evidence_audit": pd.DataFrame(evidence_audit), "model_runs": pd.DataFrame(model_runs)}
    except Exception as exc:
        # A genuine unexpected fault is returned as a succinct, inspectable
        # delivery state. It does not surface the prior notebooks' cascading
        # nested `run_ps3_production` tracebacks.
        detail = ps3v21_error_detail(exc)
        ps3v21_stage("unexpected_error", "failed", detail)
        try:
            result = ps3v21_publish_control_only(run_id, PS3V21_STATE.get("stage_audit", []), "completed_with_unexpected_error", detail)
            result["traceback"] = traceback.format_exc(limit=6)
            return result
        except Exception:
            return {"status": "completed_with_unexpected_error", "run_id": run_id, "detail": detail, "traceback": traceback.format_exc(limit=6)}


In [9]:
# 8. Run PS3 V21.
results = run_ps3_v21()
print("PS3 V21 status:", results["status"])
print("Run ID:", results["run_id"])

if "summary" in results:
    display(pd.DataFrame([results["summary"]]))
if "preflight" in results:
    display(results["preflight"])
if "source_audit" in results:
    display(results["source_audit"])
if "model_runs" in results:
    display(results["model_runs"])
if "publication" in results:
    display(pd.DataFrame(results["publication"]))

# Useful dashboard heads when a fact lane completed.
# V23: the audit tables print on EVERY path. A V22 run failed and showed no
# reason because this list held only the fact tables, so the one table that
# explained the failure was never displayed.
for table_name in ["ps3_oos_source_audit", "ps3_run_stage_audit", "ps3_root_cause_evidence_audit",
                   "ps3_source_column_profile",
                   "ps3_run_status", "ps3_device_episode_fact", "ps3_device_day", "ps3_device_reliability",
                   "ps3_serial_reliability", "ps3_facility_rollup", "ps3_repeat_interval", "ps3_commanded_split",
                   "ps3_causal_effects", "ps3_causal_balance", "ps3_device_summary", "ps3_component_summary",
                   "ps3_label_maturity", "ps3_model_scorecard", "ps3_prediction_explainability"]:
    if table_name in results.get("tables", {}):
        print(f"\n--- {table_name} ---")
        display(results["tables"][table_name].head(20))


/tmp/ipykernel_369/2581166068.py:209: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  facts["component_attribution"] = facts["component_attribution"].fillna(facts["linked_component"])


PS3 V21 status: completed
Run ID: 6a787954-9c03-422b-8f9d-ad7d7efd4f8d


,status,source_mode,oos_episode_rows,model_runs,confirmed_root_cause_rows,observed_severity_rows,validator_oos_rows,commanded_signal_rows,failure_only_rows,episode_scope_status,run_mode,historical_snapshot,data_as_of_date
0,completed,arrow_direct_silver_bootstrap,54239,"[{'target': 'observed_severity_label', 'model_...",0,0,10126,0,54239,complete_lookback_window,PRODUCTION,True,2026-04-11


,check,status,detail,reference
0,pandas_numpy,passed,None,NaN
1,scikit_learn,passed,None,NaN
2,boto3,passed,None,NaN
3,pyarrow,passed,None,NaN
4,spark_runtime,not_probed,The JVM-free pyarrow route is preferred. Spark...,None
5,ps1_oos_fact_source,passed,Source route: JVM-free pyarrow direct Silver b...,s3://cubic-mars-pm-s3-datalake-dev-gold-170202...
6,ps3_gold_label_lane,optional_not_configured,Optional label/evidence enrichment only; it is...,None
7,s3_output_prefix,passed,Immutable table/run partition output on the ar...,s3://cubic-mars-pm-s3-datalake-dev-artifacts-1...


,source,status,reference,detail,engine,pushed_down,scanned_rows,window_start,window_end,current_device_filter,rows_removed_by_current_filter,selected_rows
0,arrow_direct_silver_bootstrap,loaded,s3://cubic-mars-pm-s3-datalake-dev-gold-170202...,PS1-compatible: hardware OOS=true AND state=SE...,pyarrow_streaming,"is_hardware_oos_event, EVENT_STATE_TYPE_NAME=S...",15019419,2024-04-11,2026-04-11,applied,7406503,7612916


...

,mars_device_category,oos_episode_count,observed_severity_count,confirmed_root_cause_count,candidate_root_cause_count,component_attribution_count,severity_coverage,confirmed_root_cause_coverage
0,GATE,38448,0,0,0,38448,0.0,0.0
1,TVM,5665,0,0,0,5665,0.0,0.0
2,VALIDATOR,10126,0,0,0,10126,0.0,0.0



--- ps3_model_scorecard ---


,target,candidate_model,f1_macro,f1_weighted,balanced_accuracy,accuracy,mcc,majority_f1_macro,macro_f1_lift,label_coverage,quality_gate,detail,model_scope
0,component_attribution,ExtraTrees_balanced,0.545015,0.769683,0.614429,0.759467,0.615996,0.184976,0.360039,1.000000,passed,Temporal split at 2025-08-18 02:09:15; support...,GATE
1,component_attribution,RandomForest_balanced,0.582593,0.812378,0.608688,0.800770,0.658655,0.184976,0.397617,1.000000,passed,Temporal split at 2025-08-18 02:09:15; support...,GATE
2,component_attribution,ExtraTrees_balanced,0.196055,0.463087,0.203181,0.484827,0.241679,0.083891,0.112164,1.000000,not_passed,Temporal split at 2025-10-31 03:34:22; support...,TVM
3,component_attribution,RandomForest_balanced,0.196188,0.470238,0.211433,0.515173,0.271448,0.083891,0.112297,1.000000,not_passed,Temporal split at 2025-10-31 03:34:22; support...,TVM
4,component_attribution,ExtraTrees_balanced,0.516562,0.965869,0.513956,0.969194,0.034765,0.495014,0.021548,0.999901,not_passed,Temporal split at 2025-12-03 08:28:35; support...,VALIDATOR
5,component_attribution,RandomForest_balanced,0.495014,0.970478,0.500000,0.980253,0.000000,0.495014,0.000000,0.999901,not_passed,Temporal split at 2025-12-03 08:28:35; support...,VALIDATOR
6,component_attribution,ExtraTrees_balanced,0.244509,0.828759,0.261378,0.821935,0.527219,0.112016,0.132493,1.000000,not_passed,Temporal split at 2025-11-14 11:00:33.500000; ...,POOLED_FALLBACK
7,component_attribution,RandomForest_balanced,0.222254,0.834128,0.244643,0.845998,0.559181,0.112016,0.110238,1.000000,not_passed,Temporal split at 2025-11-14 11:00:33.500000; ...,POOLED_FALLBACK



--- ps3_prediction_explainability ---


,target,model_output,oos_episode_id,device_id,mars_device_category,predicted_label,feature,shap_value,abs_shap_value,explanation_status,explanation_note,model_scope
0,observed_severity_label,severity,(not_computed),<NA>,<NA>,<NA>,(not_computed),NaN,NaN,not_computed_model_not_publishable,SHAP is calculated only for a quality-gated mo...,NaN
1,confirmed_root_cause_domain,root_cause,(not_computed),<NA>,<NA>,<NA>,(not_computed),NaN,NaN,not_computed_model_not_publishable,SHAP is calculated only for a quality-gated mo...,NaN
2,component_attribution,component,(not_computed),<NA>,<NA>,<NA>,(not_computed),NaN,NaN,not_computed_dependency_unavailable,Install/import shap to calculate model-driver ...,GATE


## Dashboard use and result interpretation

The principal table is `ps3_device_episode_fact`, one row per OOS episode and
device, carrying both the facts and their honest semantic status.

| Field family | Use it for | Interpretation |
|---|---|---|
| `episode_*`, `oos_set_event_count` | OOS history and device drill-down | PS1-compatible observed hardware-OOS SET episode facts |
| `contains_commanded_oos_signal` | Separating planned work from failure | Commanded OOS stays in the spine so PS3 reconciles with PS1 and PS2; filter on this, do not re-cut the spine |
| `observed_severity_label`, `severity_status` | Severity reporting | Observed only when linked from PS3 evidence; blank means no label exists |
| `confirmed_root_cause_*`, `root_cause_status` | Root-cause reporting | Confirmed only with evidence plus approved taxonomy |
| `component_attribution*` | Component views | Observed or linked attribution; not automatically a confirmed root cause |
| `predicted_*`, `*_model_status` | Model estimate views | Present only after per-category temporal macro-F1 and majority-baseline gates pass |
| `dashboard_*` | Dashboard display | Preserves confirmation status instead of overwriting it with an estimate |
| `episode_scope_status` | Coverage honesty | `truncated_to_most_recent_window` means the run did not cover the full lookback |
| `freshness_status`, `is_current_operational_score` | The as-of caveat | Surface these on the screen; do not drop them |

### The tables, and what each screen should read

| Table | Screen |
|---|---|
| `ps3_run_status` | The health check. One row per table, one coherent run id. Read this before trusting anything else |
| `ps3_device_episode_fact` | Episode drill-down |
| `ps3_device_day` | The timeline. Keyed on `event_date`, the same calendar grain as PS1 `transit_day` and PS2 `event_date`, so Device 360 can draw one line |
| `ps3_device_reliability` | Device 360 header and the worst-devices list |
| `ps3_serial_reliability` | Device 360 component panel |
| `ps3_facility_rollup` | Where it happens |
| `ps3_component_summary`, `ps3_repeat_interval` | Components and what follows them |
| `ps3_causal_effects`, `ps3_causal_balance` | What actually drives repeats. Trust `significant_95_holm`, not `significant_95` |
| `ps3_label_maturity` | How much is confirmed rather than estimated |
| `ps3_model_scorecard`, `ps3_prediction_explainability` | How we know |

### Reading the causal table responsibly

`average_treatment_effect` is the adjusted difference in the probability of a
repeat OOS episode within the horizon, for episodes attributed to that component
versus comparable episodes. It is cross-fitted over time folds, so a later period
never explains an earlier one. It is observational: confounding by unmeasured
maintenance policy is not ruled out. Read `overlap_share` (below 0.80 the
comparison is weak), the balance table, and `significant_95_holm` rather than the
raw flag -- with a dozen components, one or two will clear a naive 95% bound by
chance alone.

### Expected outcomes

- Silver works but labels are absent: OOS fact, device, serial, facility and
  timeline tables all publish. Cause and severity stay explicitly unavailable.
- Evidence and approved taxonomy exist: confirmed root cause and observed
  severity populate at episode grain.
- Labels have temporal support and beat the majority baseline per category:
  estimates and SHAP publish for that category. Otherwise the scorecard says why.
- All S3 paths are immutable per `computed_date` and `run_id`. `run_complete.json`
  is written last, only for a PRODUCTION run in which every table published.
